# PlantWild Scratch Comparison

Random initialization under the locked 105-class grouped-validation protocol. Select the model and run mode in the configuration cell.


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "42"

# Change only MODEL_KEY and RUN_MODE for normal use.
SUPPORTED_MODEL_KEYS = ('wavestarnet_big_transfer_v2', 'mobilenetv3_large_100', 'efficientnet_b0')
MODEL_KEY = "wavestarnet_big_transfer_v2"
RUN_MODE = "development"

PROTOCOL_ID = "plantwild_scratch_comparison_v1"
SEEDS = [42, 2024, 3407]
MAX_EPOCHS = 100
WARMUP_EPOCHS = 5
LEARNING_RATE = 0.0003
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.05
GRADIENT_CLIP_NORM = 1.0
GRADIENT_ACCUMULATION_STEPS = 1
EARLY_STOPPING_PATIENCE = 30
EARLY_STOPPING_MIN_DELTA = 0.001
EARLY_STOPPING_START_EPOCH = 10

EXPECTED_PARAMETER_COUNTS = {'wavestarnet_big_transfer_v2': 3013109, 'mobilenetv3_large_100': 4336537, 'efficientnet_b0': 4142053}
DISPLAY_NAMES = {'wavestarnet_big_transfer_v2': 'WavestarNet-Big-Transfer-V2', 'mobilenetv3_large_100': 'MobileNetV3-Large', 'efficientnet_b0': 'EfficientNet-B0'}
MODEL_SAFE_NAMES = {'wavestarnet_big_transfer_v2': 'wavestarnet_transfer_v2', 'mobilenetv3_large_100': 'mobilenetv3_large', 'efficientnet_b0': 'efficientnet_b0'}
if MODEL_KEY not in SUPPORTED_MODEL_KEYS:
    raise ValueError(f"Unsupported MODEL_KEY: {MODEL_KEY}")
EXPECTED_PARAMETER_COUNT = EXPECTED_PARAMETER_COUNTS[MODEL_KEY]
DISPLAY_NAME = DISPLAY_NAMES[MODEL_KEY]
EXPECTED_ARCHITECTURE_SHA256 = '9ecf1f23cfdfb2c710ba89cc9f1620337ef96cbdc11f48a7c8c305df8798f474'
EXPECTED_BACKBONE_SCHEMA_SHA256 = 'f642f9b1eb6ca21454566ae81d7bc76782bed77e9210b4c4edaa32c63596bf94'
EXPECTED_VERSION = 1
KERNEL_ID = 'sayemahmedshayeed/plantwild-scratch-comparison'
OUTPUT_SAFE_NAME = "plantwild_scratch_{}_{}_v{:02d}".format(
    MODEL_SAFE_NAMES[MODEL_KEY], RUN_MODE, EXPECTED_VERSION
)
EXPECTED_MODEL_SOURCE_SHA256 = '1e600b84c2a901640ff93ec9ef8e255291cbb07f335d5217988ffe147c2ca53f'
EXPECTED_CORE_SOURCE_SHA256 = '79056ec95256ad8780776ded74594e5cbd3c62634059441d68107ec234450568'
EXPECTED_RUNTIME_SOURCE_SHA256 = '34644ef3c9e55e64d21b4fe10d3cb08f271259897d5b2454052ffd45c1477b9c'
EXPECTED_MANIFEST_SHA256 = {'train': '41fcfd247c504f0f0bdf5a0a47029436620145818a263be728af22c07b3bc84e', 'validation': '72d10da4d023c08632eb8df5ad76f566f276d07a93046950ecfc06db0fa22f87', 'class_mapping': 'aa5c417832ff1fd0d1024e085002b1e3811a46c9af64a534e0dc1fa6355d86ca'}


In [2]:
"""Canonical, self-contained WavestarNet architecture family."""

from __future__ import annotations

import hashlib
import json
from collections.abc import Mapping
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F


E11_V1_CHECKPOINT_SHA256 = (
    "9d305718b37a02fa075df80b473812dd6874d4102fec1b6b7442e1938394fd75"
)
TRANSFER_V2_PARAMETER_ELEMENTS = 2_941_100
V1_ENCODER_PARAMETER_ELEMENTS = 4_422_748


def _autocast_disabled(x: torch.Tensor):
    if x.is_cuda:
        return torch.amp.autocast(device_type="cuda", enabled=False)
    return nullcontext()


def _finite_clamp(x: torch.Tensor, value: float) -> torch.Tensor:
    return torch.nan_to_num(x, nan=0.0, posinf=value, neginf=-value).clamp(-value, value)


def _validate_probability(value: float, name: str, *, inclusive_one: bool) -> float:
    value = float(value)
    upper_ok = value <= 1.0 if inclusive_one else value < 1.0
    if value < 0.0 or not upper_ok:
        boundary = "[0, 1]" if inclusive_one else "[0, 1)"
        raise ValueError(f"{name} must be in {boundary}")
    return value


class DropPath(nn.Module):
    def __init__(self, probability: float = 0.0) -> None:
        super().__init__()
        self.probability = _validate_probability(
            probability, "DropPath probability", inclusive_one=False
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.probability == 0.0 or not self.training:
            return x
        keep = 1.0 - self.probability
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = x.new_empty(shape).bernoulli_(keep)
        return x * mask / keep


class Lifting1D(nn.Module):
    def __init__(self, channels: int, kernel_size: int = 3) -> None:
        super().__init__()
        if channels <= 0 or kernel_size <= 0 or kernel_size % 2 == 0:
            raise ValueError("channels and odd kernel_size must be positive")
        pad = kernel_size // 2
        self.predict = nn.Conv1d(
            channels, channels, kernel_size, padding=pad, groups=channels, bias=False
        )
        self.update = nn.Conv1d(
            channels, channels, kernel_size, padding=pad, groups=channels, bias=False
        )
        with torch.no_grad():
            self.predict.weight.zero_()
            self.update.weight.zero_()
            self.predict.weight[:, :, pad] = 1.0
            self.update.weight[:, :, pad] = 0.5

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        if x.shape[-1] % 2 == 1:
            x = F.pad(x, (0, 1), mode="replicate")
        even, odd = x[..., 0::2], x[..., 1::2]
        detail = odd - self.predict(even)
        approximation = even + self.update(detail)
        return approximation, detail

    def inverse(
        self, approximation: torch.Tensor, detail: torch.Tensor, out_len: int | None = None
    ) -> torch.Tensor:
        even = approximation - self.update(detail)
        odd = detail + self.predict(even)
        output = torch.stack((even, odd), dim=-1).flatten(-2)
        return output if out_len is None else output[..., :out_len]


class DWT2D(nn.Module):
    def __init__(self, channels: int, kernel_size: int = 3) -> None:
        super().__init__()
        self.lw = Lifting1D(channels, kernel_size)
        self.lh = Lifting1D(channels, kernel_size)

    def _width(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        batch, channels, height, width = x.shape
        rows = x.permute(0, 2, 1, 3).contiguous().reshape(batch * height, channels, width)
        low, high = self.lw(rows)
        low = low.reshape(batch, height, channels, -1).permute(0, 2, 1, 3).contiguous()
        high = high.reshape(batch, height, channels, -1).permute(0, 2, 1, 3).contiguous()
        return low, high

    def _width_inverse(
        self, low: torch.Tensor, high: torch.Tensor, out_width: int | None
    ) -> torch.Tensor:
        batch, channels, height, half_width = low.shape
        low_rows = low.permute(0, 2, 1, 3).contiguous().reshape(
            batch * height, channels, half_width
        )
        high_rows = high.permute(0, 2, 1, 3).contiguous().reshape(
            batch * height, channels, half_width
        )
        output = self.lw.inverse(low_rows, high_rows, out_width)
        return output.reshape(batch, height, channels, -1).permute(0, 2, 1, 3).contiguous()

    def _height(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        batch, channels, height, width = x.shape
        columns = x.permute(0, 3, 1, 2).contiguous().reshape(
            batch * width, channels, height
        )
        low, high = self.lh(columns)
        low = low.reshape(batch, width, channels, -1).permute(0, 2, 3, 1).contiguous()
        high = high.reshape(batch, width, channels, -1).permute(0, 2, 3, 1).contiguous()
        return low, high

    def _height_inverse(
        self, low: torch.Tensor, high: torch.Tensor, out_height: int | None
    ) -> torch.Tensor:
        batch, channels, half_height, width = low.shape
        low_columns = low.permute(0, 3, 1, 2).contiguous().reshape(
            batch * width, channels, half_height
        )
        high_columns = high.permute(0, 3, 1, 2).contiguous().reshape(
            batch * width, channels, half_height
        )
        output = self.lh.inverse(low_columns, high_columns, out_height)
        return output.reshape(batch, width, channels, -1).permute(0, 2, 3, 1).contiguous()

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, ...]:
        low, high = self._width(x)
        ll, lh = self._height(low)
        hl, hh = self._height(high)
        return ll, lh, hl, hh

    def inverse(
        self,
        ll: torch.Tensor,
        lh: torch.Tensor,
        hl: torch.Tensor,
        hh: torch.Tensor,
        out_hw: tuple[int, int] | None = None,
    ) -> torch.Tensor:
        out_height = out_width = None
        if out_hw is not None:
            out_height, out_width = out_hw
        low = self._height_inverse(ll, lh, out_height)
        high = self._height_inverse(hl, hh, out_height)
        return self._width_inverse(low, high, out_width)


class WaveletGate(nn.Module):
    def __init__(self, channels: int, kernel_size: int = 3, reduction: int = 8) -> None:
        super().__init__()
        self.dwt = DWT2D(channels, kernel_size)
        hidden = max(channels // reduction, 8)
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels * 4, hidden, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels * 4, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output_dtype = x.dtype
        with _autocast_disabled(x):
            x32 = x.float()
            bands = torch.cat(self.dwt(x32), dim=1)
            gate = torch.nan_to_num(
                self.gate(bands), nan=0.5, posinf=1.0, neginf=0.0
            ).clamp(1e-4, 1.0)
            bands = _finite_clamp(bands * gate, 20.0)
            output = self.dwt.inverse(
                *torch.split(bands, x32.shape[1], dim=1), out_hw=x32.shape[2:]
            )
        return _finite_clamp(output, 20.0).to(output_dtype)


class S3StarBlock(nn.Module):
    def __init__(
        self,
        channels: int,
        dw_kernel: int = 7,
        expand: int = 2,
        layerscale: float = 1e-4,
        drop_path: float = 0.0,
    ) -> None:
        super().__init__()
        expanded = channels * expand
        self.dw = nn.Conv2d(
            channels, channels, dw_kernel, padding=dw_kernel // 2,
            groups=channels, bias=False,
        )
        self.bn_s = nn.BatchNorm2d(channels)
        self.spec = WaveletGate(channels)
        self.bn_f = nn.BatchNorm2d(channels)
        self.proj_s = nn.Conv2d(channels, expanded, 1, bias=False)
        self.proj_f = nn.Conv2d(channels, expanded, 1, bias=False)
        self.pbn_s = nn.BatchNorm2d(expanded)
        self.pbn_f = nn.BatchNorm2d(expanded)
        self.act = nn.Hardswish(inplace=True)
        self.proj_out = nn.Conv2d(expanded, channels, 1, bias=False)
        self.bn_out = nn.BatchNorm2d(channels)
        self.gamma = nn.Parameter(layerscale * torch.ones(1, channels, 1, 1))
        self.drop_path = DropPath(drop_path)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        spatial = self.bn_s(self.dw(x))
        spectral = self.bn_f(self.spec(x))
        a = self.act(self.pbn_s(self.proj_s(spatial)))
        b = self.act(self.pbn_f(self.proj_f(spectral)))
        with _autocast_disabled(a):
            fused = _finite_clamp(a.float() * b.float(), 8.0).to(a.dtype)
        output = self.gamma * self.bn_out(self.proj_out(fused))
        return x + self.drop_path(output)


class TransferS3StarBlock(nn.Module):
    """E11-compatible S3 block with an optional lightweight spectral bypass."""

    def __init__(
        self,
        channels: int,
        use_wavelet: bool,
        dw_kernel: int = 7,
        expand: int = 2,
        layerscale: float = 1e-4,
        drop_path: float = 0.0,
    ) -> None:
        super().__init__()
        expanded = channels * expand
        self.dw = nn.Conv2d(
            channels,
            channels,
            dw_kernel,
            padding=dw_kernel // 2,
            groups=channels,
            bias=False,
        )
        self.bn_s = nn.BatchNorm2d(channels)
        if use_wavelet:
            self.spec = WaveletGate(channels)
        self.bn_f = nn.BatchNorm2d(channels)
        self.proj_s = nn.Conv2d(channels, expanded, 1, bias=False)
        self.proj_f = nn.Conv2d(channels, expanded, 1, bias=False)
        self.pbn_s = nn.BatchNorm2d(expanded)
        self.pbn_f = nn.BatchNorm2d(expanded)
        self.act = nn.Hardswish(inplace=True)
        self.proj_out = nn.Conv2d(expanded, channels, 1, bias=False)
        self.bn_out = nn.BatchNorm2d(channels)
        self.gamma = nn.Parameter(layerscale * torch.ones(1, channels, 1, 1))
        self.drop_path = DropPath(drop_path)

    def spectral_features(self, x: torch.Tensor) -> torch.Tensor:
        if hasattr(self, "spec"):
            return self.spec(x)
        return x * 0.5

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        spatial = self.bn_s(self.dw(x))
        spectral = self.bn_f(self.spectral_features(x))
        a = self.act(self.pbn_s(self.proj_s(spatial)))
        b = self.act(self.pbn_f(self.proj_f(spectral)))
        with _autocast_disabled(a):
            fused = _finite_clamp(a.float() * b.float(), 8.0).to(a.dtype)
        output = (self.gamma * self.bn_out(self.proj_out(fused))).to(x.dtype)
        return x + self.drop_path(output)


class WaveDown(nn.Module):
    def __init__(self, input_channels: int, output_channels: int) -> None:
        super().__init__()
        self.dwt = DWT2D(input_channels, 3)
        self.proj = nn.Conv2d(input_channels * 4, output_channels, 1, bias=False)
        self.bn = nn.BatchNorm2d(output_channels)
        self.act = nn.Hardswish(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output_dtype = x.dtype
        with _autocast_disabled(x):
            bands = _finite_clamp(torch.cat(self.dwt(x.float()), dim=1), 20.0)
        return self.act(self.bn(self.proj(bands.to(output_dtype))))


class WavestarNetV1(nn.Module):
    def __init__(self, num_classes: int, dropout: float, max_drop_path: float) -> None:
        super().__init__()
        stem_channels = 24
        widths = (48, 96, 192, 384)
        depths = (2, 3, 6, 2)
        self.stem = nn.Sequential(
            nn.Conv2d(3, stem_channels, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(stem_channels),
            nn.Hardswish(inplace=True),
        )
        stages: list[nn.Module] = []
        input_channels = stem_channels
        block_index = 0
        total_blocks = sum(depths)
        for output_channels, depth in zip(widths, depths):
            stage: list[nn.Module] = [WaveDown(input_channels, output_channels)]
            for _ in range(depth):
                probability = max_drop_path * block_index / max(1, total_blocks - 1)
                stage.append(S3StarBlock(output_channels, drop_path=probability))
                block_index += 1
            stages.append(nn.Sequential(*stage))
            input_channels = output_channels
        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(widths[-1], 1536),
            nn.Hardswish(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(1536, num_classes),
        )

    def forward_features(self, x: torch.Tensor) -> tuple[torch.Tensor, ...]:
        x = self.stem(x)
        features = []
        for stage in self.stages:
            x = stage(x)
            features.append(x)
        return tuple(features)

    def forward_head(self, features: tuple[torch.Tensor, ...]) -> torch.Tensor:
        if len(features) != 4:
            raise ValueError("forward_head requires four stage features")
        return self.head(features[-1])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.forward_head(self.forward_features(x))

    def classifier_module(self) -> nn.Module:
        return self.head


class WavestarNetBigTransferV2(nn.Module):
    """Compact V1-compatible model designed for exact E11 weight transfer."""

    WIDTHS = (48, 96, 192, 384)
    DEPTHS = (2, 3, 5, 1)
    MASKS = ((1, 0), (1, 0, 0), (1, 0, 0, 0, 1), (1,))

    def __init__(self, num_classes: int, dropout: float, max_drop_path: float) -> None:
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 24, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(24),
            nn.Hardswish(inplace=True),
        )
        rates = torch.linspace(0, max_drop_path, sum(self.DEPTHS)).tolist()
        stages: list[nn.Module] = []
        input_channels = 24
        rate_index = 0
        for output_channels, depth, mask in zip(self.WIDTHS, self.DEPTHS, self.MASKS):
            stage: list[nn.Module] = [WaveDown(input_channels, output_channels)]
            for block_index in range(depth):
                stage.append(
                    TransferS3StarBlock(
                        output_channels,
                        use_wavelet=bool(mask[block_index]),
                        drop_path=rates[rate_index],
                    )
                )
                rate_index += 1
            stages.append(nn.Sequential(*stage))
            input_channels = output_channels
        self.stages = nn.Sequential(*stages)
        pooled_channels = sum(self.WIDTHS[1:])
        self.head = nn.Sequential(
            nn.LayerNorm(pooled_channels),
            nn.Dropout(dropout),
            nn.Linear(pooled_channels, num_classes),
        )

    def forward_features(self, x: torch.Tensor) -> tuple[torch.Tensor, ...]:
        x = self.stem(x)
        features = []
        for stage in self.stages:
            x = stage(x)
            features.append(x)
        return tuple(features)

    def forward_head(self, features: tuple[torch.Tensor, ...]) -> torch.Tensor:
        if len(features) != 4:
            raise ValueError("forward_head requires four stage features")
        pooled = torch.cat(
            tuple(feature.float().mean(dim=(-2, -1)) for feature in features[1:]),
            dim=1,
        )
        pooled = self.head[0](pooled).to(features[-1].dtype)
        return self.head[2](self.head[1](pooled))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.forward_head(self.forward_features(x))

    def classifier_module(self) -> nn.Module:
        return self.head[2]


class WaveStem(nn.Module):
    def __init__(self, output_channels: int, wavelet_clamp: float = 20.0) -> None:
        super().__init__()
        self.dwt = DWT2D(3)
        self.mix = nn.Sequential(
            nn.Conv2d(12, output_channels, 1, bias=False),
            nn.BatchNorm2d(output_channels),
            nn.Hardswish(inplace=True),
            nn.Conv2d(
                output_channels, output_channels, 3, padding=1,
                groups=output_channels, bias=False,
            ),
            nn.BatchNorm2d(output_channels),
            nn.Hardswish(inplace=True),
        )
        self.wavelet_clamp = float(wavelet_clamp)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output_dtype = x.dtype
        with _autocast_disabled(x):
            bands = _finite_clamp(torch.cat(self.dwt(x.float()), dim=1), self.wavelet_clamp)
        return self.mix(bands.to(output_dtype))


class ConfiguredWaveDown(nn.Module):
    def __init__(
        self, input_channels: int, output_channels: int, wavelet_clamp: float = 20.0
    ) -> None:
        super().__init__()
        self.dwt = DWT2D(input_channels)
        self.proj = nn.Conv2d(4 * input_channels, output_channels, 1, bias=False)
        self.bn = nn.BatchNorm2d(output_channels)
        self.act = nn.Hardswish(inplace=True)
        self.wavelet_clamp = float(wavelet_clamp)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output_dtype = x.dtype
        with _autocast_disabled(x):
            bands = _finite_clamp(torch.cat(self.dwt(x.float()), dim=1), self.wavelet_clamp)
        return self.act(self.bn(self.proj(bands.to(output_dtype))))


class IdentityWaveletGate(nn.Module):
    def __init__(
        self, channels: int, reduction: int = 8, wavelet_clamp: float = 20.0
    ) -> None:
        super().__init__()
        self.dwt = DWT2D(channels)
        hidden = max(channels // reduction, 8)
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(4 * channels, hidden, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, 4 * channels, 1),
        )
        self.wavelet_clamp = float(wavelet_clamp)
        nn.init.zeros_(self.gate[-1].weight)
        nn.init.zeros_(self.gate[-1].bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output_dtype = x.dtype
        with _autocast_disabled(x):
            x32 = x.float()
            bands = torch.cat(self.dwt(x32), dim=1)
            gain = 1.0 + 0.5 * torch.tanh(self.gate(bands))
            gated = _finite_clamp(bands * gain, self.wavelet_clamp)
            output = self.dwt.inverse(
                *gated.chunk(4, dim=1), out_hw=(x32.shape[-2], x32.shape[-1])
            )
            output = _finite_clamp(output, self.wavelet_clamp)
        return output.to(output_dtype)


class PointwiseBNAct(nn.Sequential):
    def __init__(self, input_channels: int, output_channels: int) -> None:
        super().__init__(
            nn.Conv2d(input_channels, output_channels, 1, bias=False),
            nn.BatchNorm2d(output_channels),
            nn.Hardswish(inplace=True),
        )


class SelectiveWaveStarBlock(nn.Module):
    def __init__(
        self,
        channels: int,
        use_wavelet: bool,
        kernel: int,
        layer_scale: float,
        drop_path: float,
        fuse_clamp: float = 8.0,
    ) -> None:
        super().__init__()
        self.spatial = nn.Sequential(
            nn.Conv2d(
                channels, channels, kernel, padding=kernel // 2,
                groups=channels, bias=False,
            ),
            nn.BatchNorm2d(channels),
        )
        self.spectral = IdentityWaveletGate(channels) if use_wavelet else None
        self.bn_f = nn.BatchNorm2d(channels)
        self.proj_a = PointwiseBNAct(channels, 2 * channels)
        self.proj_b = PointwiseBNAct(channels, 2 * channels)
        self.out = nn.Sequential(
            nn.Conv2d(2 * channels, channels, 1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.gamma = nn.Parameter(layer_scale * torch.ones(1, channels, 1, 1))
        self.drop_path = DropPath(drop_path)
        self.fuse_clamp = float(fuse_clamp)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a = self.proj_a(self.spatial(x))
        spectral = self.spectral(x) if self.spectral is not None else x
        b = self.proj_b(self.bn_f(spectral))
        with _autocast_disabled(a):
            fused = _finite_clamp(a.float() * b.float(), self.fuse_clamp).to(a.dtype)
        residual = self.gamma * self.out(fused)
        return x + self.drop_path(residual)


def _avg_rms_pool(x: torch.Tensor) -> torch.Tensor:
    x32 = x.float()
    average = x32.mean(dim=(-2, -1))
    rms = x32.square().mean(dim=(-2, -1)).add(1e-6).sqrt()
    return torch.cat((average, rms), dim=1)


class SelectiveWaveStarNet(nn.Module):
    def __init__(
        self,
        num_classes: int,
        stem_channels: int,
        widths: tuple[int, ...],
        depths: tuple[int, ...],
        kernels: tuple[int, ...],
        masks: tuple[tuple[int, ...], ...],
        head_dim: int,
        dropout: float,
        max_drop_path: float,
        layer_scale: float = 1e-3,
    ) -> None:
        super().__init__()
        if not (len(widths) == len(depths) == len(kernels) == len(masks) == 4):
            raise ValueError("widths, depths, kernels, and masks must have four stages")
        if any(value <= 0 for value in (*widths, *depths, *kernels)):
            raise ValueError("channel, depth, and kernel values must be positive")
        if any(len(mask) != depth for mask, depth in zip(masks, depths)):
            raise ValueError("each wavelet mask length must equal its stage depth")
        if any(value not in (0, 1) for mask in masks for value in mask):
            raise ValueError("wavelet masks must contain only zero or one")
        rates = torch.linspace(0, max_drop_path, sum(depths)).tolist()
        self.stem = WaveStem(stem_channels)
        self.stages = nn.ModuleList()
        rate_id = 0
        input_channels = stem_channels
        for output_channels, depth, kernel, mask in zip(widths, depths, kernels, masks):
            blocks: list[nn.Module] = [ConfiguredWaveDown(input_channels, output_channels)]
            for block_id in range(depth):
                blocks.append(
                    SelectiveWaveStarBlock(
                        output_channels,
                        bool(mask[block_id]),
                        kernel,
                        layer_scale,
                        rates[rate_id],
                    )
                )
                rate_id += 1
            self.stages.append(nn.Sequential(*blocks))
            input_channels = output_channels
        pooled_dim = 2 * sum(widths[1:])
        self.head_norm = nn.LayerNorm(pooled_dim)
        self.head_fc1 = nn.Linear(pooled_dim, head_dim)
        self.head_act = nn.Hardswish(inplace=True)
        self.head_drop = nn.Dropout(dropout)
        self.head_fc2 = nn.Linear(head_dim, num_classes)

    def forward_features(self, x: torch.Tensor) -> tuple[torch.Tensor, ...]:
        x = self.stem(x)
        features = []
        for stage in self.stages:
            x = stage(x)
            features.append(x)
        return tuple(features)

    def forward_head(self, features: tuple[torch.Tensor, ...]) -> torch.Tensor:
        if len(features) != 4:
            raise ValueError("forward_head requires four stage features")
        pooled = torch.cat(tuple(_avg_rms_pool(x) for x in features[1:]), dim=1)
        pooled = self.head_norm(pooled).to(features[-1].dtype)
        pooled = self.head_drop(self.head_act(self.head_fc1(pooled)))
        return self.head_fc2(pooled)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.forward_head(self.forward_features(x))

    def classifier_module(self) -> nn.Module:
        return self.head_fc2


@dataclass(frozen=True)
class WavestarNetSpec:
    key: str
    display_name: str
    stem_channels: int
    widths: tuple[int, int, int, int]
    depths: tuple[int, int, int, int]
    kernels: tuple[int, int, int, int]
    masks: tuple[tuple[int, ...], ...]
    head_dim: int
    dropout: float
    max_drop_path: float
    layer_scale: float
    supports_e11: bool


MODEL_REGISTRY: dict[str, WavestarNetSpec] = {
    "wavestarnet_big_v1": WavestarNetSpec(
        "wavestarnet_big_v1", "WavestarNet-Big V1", 24,
        (48, 96, 192, 384), (2, 3, 6, 2), (7, 7, 7, 7),
        ((1, 1), (1, 1, 1), (1, 1, 1, 1, 1, 1), (1, 1)),
        1536, 0.25, 0.0, 1e-4, True,
    ),
    "wavestarnet_big_acc": WavestarNetSpec(
        "wavestarnet_big_acc", "WavestarNet-Big-Acc", 24,
        (48, 96, 192, 384), (2, 4, 6, 1), (7, 7, 7, 7),
        ((1, 0), (1, 0, 0, 1), (1, 0, 0, 0, 0, 1), (1,)),
        448, 0.20, 0.10, 1e-3, False,
    ),
    "wavestarnet_big_lite": WavestarNetSpec(
        "wavestarnet_big_lite", "WavestarNet-Big-Lite", 20,
        (40, 80, 160, 256), (2, 3, 5, 1), (5, 5, 7, 7),
        ((1, 0), (1, 0, 0), (1, 0, 0, 0, 0), (1,)),
        256, 0.15, 0.08, 1e-3, False,
    ),
    "wavestarnet_big_transfer_v2": WavestarNetSpec(
        "wavestarnet_big_transfer_v2", "WavestarNet-Big-Transfer-V2", 24,
        (48, 96, 192, 384), (2, 3, 5, 1), (7, 7, 7, 7),
        ((1, 0), (1, 0, 0), (1, 0, 0, 0, 1), (1,)),
        672, 0.15, 0.08, 1e-4, True,
    ),
}


def create_wavestarnet(
    model_key: str,
    num_classes: int,
    *,
    dropout: float | None = None,
    max_drop_path: float | None = None,
) -> nn.Module:
    if model_key not in MODEL_REGISTRY:
        raise ValueError(f"unknown model key: {model_key}")
    if not isinstance(num_classes, int) or num_classes <= 0:
        raise ValueError("num_classes must be a positive integer")
    spec = MODEL_REGISTRY[model_key]
    dropout_value = spec.dropout if dropout is None else _validate_probability(
        dropout, "dropout", inclusive_one=False
    )
    drop_path_value = spec.max_drop_path if max_drop_path is None else _validate_probability(
        max_drop_path, "drop path", inclusive_one=False
    )
    if model_key == "wavestarnet_big_v1":
        return WavestarNetV1(num_classes, dropout_value, drop_path_value)
    if model_key == "wavestarnet_big_transfer_v2":
        return WavestarNetBigTransferV2(num_classes, dropout_value, drop_path_value)
    return SelectiveWaveStarNet(
        num_classes=num_classes,
        stem_channels=spec.stem_channels,
        widths=spec.widths,
        depths=spec.depths,
        kernels=spec.kernels,
        masks=spec.masks,
        head_dim=spec.head_dim,
        dropout=dropout_value,
        max_drop_path=drop_path_value,
        layer_scale=spec.layer_scale,
    )


def _state_schema(model_key: str, num_classes: int, *, backbone_only: bool) -> list[dict[str, Any]]:
    model = create_wavestarnet(model_key, num_classes)
    if model_key == "wavestarnet_big_v1":
        excluded = ("head.5.",)
    elif model_key == "wavestarnet_big_transfer_v2":
        excluded = ("head.2.",)
    else:
        excluded = ("head_fc2.",)
    schema = []
    for name, value in sorted(model.state_dict().items()):
        if backbone_only and name.startswith(excluded):
            continue
        schema.append({"name": name, "shape": list(value.shape), "dtype": str(value.dtype)})
    return schema


def architecture_manifest(model_key: str, num_classes: int) -> dict[str, Any]:
    if model_key not in MODEL_REGISTRY:
        raise ValueError(f"unknown model key: {model_key}")
    if not isinstance(num_classes, int) or num_classes <= 0:
        raise ValueError("num_classes must be a positive integer")
    model = create_wavestarnet(model_key, num_classes)
    return {
        "model_key": model_key,
        "num_classes": num_classes,
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "spec": asdict(MODEL_REGISTRY[model_key]),
        "state_schema": _state_schema(model_key, num_classes, backbone_only=False),
    }


def _hash_json(payload: Any) -> str:
    encoded = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def architecture_fingerprint(model_key: str, num_classes: int) -> str:
    return _hash_json(architecture_manifest(model_key, num_classes))


def backbone_schema_fingerprint(model_key: str) -> str:
    if model_key not in MODEL_REGISTRY:
        raise ValueError(f"unknown model key: {model_key}")
    return _hash_json(
        {
            "model_key": model_key,
            "spec": asdict(MODEL_REGISTRY[model_key]),
            "state_schema": _state_schema(model_key, 21, backbone_only=True),
        }
    )


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _duplicate_encoder_aliases(model: nn.Module) -> list[list[str]]:
    aliases: list[list[str]] = []
    for values in (
        model.named_parameters(remove_duplicate=False),
        model.named_buffers(remove_duplicate=False),
    ):
        names_by_identity: dict[int, list[str]] = {}
        for name, value in values:
            if name.startswith(("stem.", "stages.")):
                names_by_identity.setdefault(id(value), []).append(name)
        aliases.extend(names for names in names_by_identity.values() if len(names) > 1)
    return aliases


def load_e11_transfer_v2(
    model: nn.Module,
    checkpoint_path: str | Path,
    *,
    expected_sha256: str = E11_V1_CHECKPOINT_SHA256,
) -> dict[str, Any]:
    """Load the exact retained V1 encoder tensors into Transfer V2, fail closed."""

    if not isinstance(model, WavestarNetBigTransferV2):
        raise ValueError("E11 Transfer V2 loader requires WavestarNet-Big-Transfer-V2")
    duplicate_aliases = _duplicate_encoder_aliases(model)
    if duplicate_aliases:
        raise ValueError(f"duplicate target encoder aliases: {duplicate_aliases}")

    path = Path(checkpoint_path)
    if not path.is_file():
        raise FileNotFoundError(f"E11 checkpoint not found: {path}")
    observed_sha256 = _sha256_file(path)
    if observed_sha256.lower() != str(expected_sha256).lower():
        raise ValueError(
            "E11 checkpoint SHA-256 mismatch: "
            f"expected {expected_sha256}, observed {observed_sha256}"
        )

    payload = torch.load(path, map_location="cpu", weights_only=True)
    if not isinstance(payload, Mapping):
        raise ValueError("E11 checkpoint must contain a tensor state mapping")
    wrappers = [key for key in ("model_state_dict", "state_dict") if key in payload]
    if len(wrappers) > 1:
        raise ValueError(f"ambiguous E11 checkpoint wrappers: {wrappers}")
    source_state = payload[wrappers[0]] if wrappers else payload
    if not isinstance(source_state, Mapping):
        raise ValueError("E11 checkpoint state must be a mapping")
    if any(not isinstance(key, str) or not torch.is_tensor(value)
           for key, value in source_state.items()):
        raise ValueError("E11 checkpoint contains non-tensor state values")

    target_state = model.state_dict()
    encoder_keys = sorted(
        key for key in target_state if key.startswith(("stem.", "stages."))
    )
    matching_keys = [key for key in encoder_keys if key in source_state]
    if not matching_keys:
        raise ValueError("no E11 tensors match the Transfer V2 encoder")
    missing_keys = sorted(set(encoder_keys) - set(source_state))
    if missing_keys:
        raise ValueError(f"unexpected target keys absent from E11: {missing_keys}")

    shape_errors = [
        (key, tuple(source_state[key].shape), tuple(target_state[key].shape))
        for key in encoder_keys
        if tuple(source_state[key].shape) != tuple(target_state[key].shape)
    ]
    if shape_errors:
        raise ValueError(f"E11 retained tensor shape mismatch: {shape_errors}")
    dtype_errors = [
        (key, str(source_state[key].dtype), str(target_state[key].dtype))
        for key in encoder_keys
        if source_state[key].dtype != target_state[key].dtype
    ]
    if dtype_errors:
        raise ValueError(f"E11 retained tensor dtype mismatch: {dtype_errors}")

    head_before = {
        key: value.detach().clone()
        for key, value in target_state.items()
        if key.startswith("head.")
    }
    updated_state = dict(target_state)
    for key in encoder_keys:
        updated_state[key] = source_state[key].detach().clone()
    model.load_state_dict(updated_state, strict=True)

    loaded_state = model.state_dict()
    exact_tensor_equality = all(
        torch.equal(loaded_state[key].cpu(), source_state[key].cpu())
        for key in encoder_keys
    )
    head_preserved = all(
        torch.equal(loaded_state[key].cpu(), value.cpu())
        for key, value in head_before.items()
    )
    if not exact_tensor_equality:
        raise RuntimeError("E11 transfer failed exact tensor equality validation")
    if not head_preserved:
        raise RuntimeError("E11 transfer modified the fresh classifier head")

    parameter_elements = {
        key: parameter.numel() for key, parameter in model.named_parameters()
    }
    loaded_parameter_elements = sum(
        parameter_elements.get(key, 0) for key in encoder_keys
    )
    if loaded_parameter_elements != TRANSFER_V2_PARAMETER_ELEMENTS:
        raise ValueError(
            "unexpected transferred parameter count: "
            f"expected {TRANSFER_V2_PARAMETER_ELEMENTS}, observed {loaded_parameter_elements}"
        )
    total_parameter_elements = sum(parameter.numel() for parameter in model.parameters())
    stage_coverage = {}
    for stage_index in range(4):
        prefix = f"stages.{stage_index}."
        target_elements = sum(
            parameter.numel()
            for name, parameter in model.named_parameters()
            if name.startswith(prefix)
        )
        loaded_elements = sum(
            parameter_elements.get(key, 0)
            for key in encoder_keys
            if key.startswith(prefix)
        )
        stage_coverage[f"stage_{stage_index + 1}"] = {
            "loaded_parameter_elements": loaded_elements,
            "target_parameter_elements": target_elements,
            "coverage": loaded_elements / target_elements,
        }
    skipped_keys = sorted(
        key for key in source_state
        if key.startswith(("stem.", "stages.")) and key not in encoder_keys
    )
    all_skipped_source_keys = sorted(set(source_state) - set(encoder_keys))
    newly_initialized_target_keys = sorted(
        key for key in target_state if key not in encoder_keys
    )
    num_classes = model.head[2].out_features
    return {
        "target_model_key": "wavestarnet_big_transfer_v2",
        "target_num_classes": num_classes,
        "target_architecture_sha256": architecture_fingerprint(
            "wavestarnet_big_transfer_v2", num_classes
        ),
        "target_backbone_schema_sha256": backbone_schema_fingerprint(
            "wavestarnet_big_transfer_v2"
        ),
        "source_checkpoint": str(path),
        "source_sha256": observed_sha256,
        "expected_sha256": str(expected_sha256).lower(),
        "loaded_keys": encoder_keys,
        "loaded_key_count": len(encoder_keys),
        "loaded_tensor_count": len(encoder_keys),
        "loaded_state_elements": sum(target_state[key].numel() for key in encoder_keys),
        "skipped_source_encoder_keys": skipped_keys,
        "skipped_source_keys": all_skipped_source_keys,
        "newly_initialized_target_keys": newly_initialized_target_keys,
        "loaded_parameter_elements": loaded_parameter_elements,
        "target_parameter_elements": total_parameter_elements,
        "target_parameter_coverage": loaded_parameter_elements / total_parameter_elements,
        "original_e11_encoder_parameter_elements": V1_ENCODER_PARAMETER_ELEMENTS,
        "original_e11_encoder_utilization": (
            loaded_parameter_elements / V1_ENCODER_PARAMETER_ELEMENTS
        ),
        "per_stage_coverage": stage_coverage,
        "exact_tensor_equality": exact_tensor_equality,
        "head_preserved": head_preserved,
    }


In [3]:
import base64
EMBEDDED_MODEL_SOURCE_BYTES = base64.b64decode('...')
EMBEDDED_CORE_SOURCE_BYTES = base64.b64decode('IiIiUHVyZSBjb250cmFjdHMgZm9yIHRoZSBjb250cm9sbGVkIFBsYW50V2lsZCBzY3JhdGNoIGNvbXBhcmlzb24uIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aApmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIEl0ZXJhYmxlLCBTZXF1ZW5jZQoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIE1vZGVsU3BlYzoKICAgIGtleTogc3RyCiAgICBkaXNwbGF5X25hbWU6IHN0cgogICAga2luZDogc3RyCiAgICB0aW1tX25hbWU6IHN0ciB8IE5vbmUKICAgIGV4cGVjdGVkX3BhcmFtZXRlcnM6IGludAogICAgcHJldHJhaW5lZDogYm9vbCA9IEZhbHNlCgoKTU9ERUxfU1BFQ1MgPSB7CiAgICAid2F2ZXN0YXJuZXRfYmlnX3RyYW5zZmVyX3YyIjogTW9kZWxTcGVjKAogICAgICAgIGtleT0id2F2ZXN0YXJuZXRfYmlnX3RyYW5zZmVyX3YyIiwKICAgICAgICBkaXNwbGF5X25hbWU9IldhdmVzdGFyTmV0LUJpZy1UcmFuc2Zlci1WMiIsCiAgICAgICAga2luZD0id2F2ZXN0YXJuZXQiLAogICAgICAgIHRpbW1fbmFtZT1Ob25lLAogICAgICAgIGV4cGVjdGVkX3BhcmFtZXRlcnM9M18wMTNfMTA5LAogICAgKSwKICAgICJtb2JpbGVuZXR2M19sYXJnZV8xMDAiOiBNb2RlbFNwZWMoCiAgICAgICAga2V5PSJtb2JpbGVuZXR2M19sYXJnZV8xMDAiLAogICAgICAgIGRpc3BsYXlfbmFtZT0iTW9iaWxlTmV0VjMtTGFyZ2UiLAogICAgICAgIGtpbmQ9InRpbW0iLAogICAgICAgIHRpbW1fbmFtZT0ibW9iaWxlbmV0djNfbGFyZ2VfMTAwIiwKICAgICAgICBleHBlY3RlZF9wYXJhbWV0ZXJzPTRfMzM2XzUzNywKICAgICksCiAgICAiZWZmaWNpZW50bmV0X2IwIjogTW9kZWxTcGVjKAogICAgICAgIGtleT0iZWZmaWNpZW50bmV0X2IwIiwKICAgICAgICBkaXNwbGF5X25hbWU9IkVmZmljaWVudE5ldC1CMCIsCiAgICAgICAga2luZD0idGltbSIsCiAgICAgICAgdGltbV9uYW1lPSJlZmZpY2llbnRuZXRfYjAiLAogICAgICAgIGV4cGVjdGVkX3BhcmFtZXRlcnM9NF8xNDJfMDUzLAogICAgKSwKfQoKCmRlZiBtb2RlbF9zcGVjKGtleTogc3RyKSAtPiBNb2RlbFNwZWM6CiAgICB0cnk6CiAgICAgICAgcmV0dXJuIE1PREVMX1NQRUNTW2tleV0KICAgIGV4Y2VwdCBLZXlFcnJvciBhcyBlcnJvcjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5rbm93biBzY3JhdGNoIG1vZGVsX2tleToge2tleX0iKSBmcm9tIGVycm9yCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgU2NyYXRjaENvbmZpZzoKICAgIG1vZGVsX2tleTogc3RyCiAgICBydW5fbW9kZTogc3RyCiAgICBzZWVkczogdHVwbGVbaW50LCAuLi5dID0gKDQyLCAyMDI0LCAzNDA3KQogICAgbnVtX2NsYXNzZXM6IGludCA9IDEwNQogICAgaW1hZ2Vfc2l6ZTogaW50ID0gMjI0CiAgICB0cmFpbl9iYXRjaF9zaXplOiBpbnQgPSAzMgogICAgZXZhbF9iYXRjaF9zaXplOiBpbnQgPSA2NAogICAgbWF4X2Vwb2NoczogaW50ID0gMTAwCiAgICBzbW9rZV9lcG9jaHM6IGludCA9IDEKICAgIHdhcm11cF9lcG9jaHM6IGludCA9IDUKICAgIGxlYXJuaW5nX3JhdGU6IGZsb2F0ID0gM2UtNAogICAgd2VpZ2h0X2RlY2F5OiBmbG9hdCA9IDAuMDUKICAgIGxhYmVsX3Ntb290aGluZzogZmxvYXQgPSAwLjA1CiAgICBncmFkaWVudF9jbGlwX25vcm06IGZsb2F0ID0gMS4wCiAgICBncmFkaWVudF9hY2N1bXVsYXRpb25fc3RlcHM6IGludCA9IDEKICAgIG1pbl9scl9yYXRpbzogZmxvYXQgPSAwLjA1CiAgICBlYXJseV9zdG9wcGluZ19wYXRpZW5jZTogaW50ID0gMzAKICAgIGVhcmx5X3N0b3BwaW5nX21pbl9kZWx0YTogZmxvYXQgPSAwLjAwMQogICAgZWFybHlfc3RvcHBpbmdfc3RhcnRfZXBvY2g6IGludCA9IDEwCgogICAgZGVmIF9fcG9zdF9pbml0X18oc2VsZikgLT4gTm9uZToKICAgICAgICBtb2RlbF9zcGVjKHNlbGYubW9kZWxfa2V5KQogICAgICAgIGFjdGl2ZV9zZWVkcyhzZWxmLnJ1bl9tb2RlLCBzZWxmLnNlZWRzKQogICAgICAgIGlmIHNlbGYubnVtX2NsYXNzZXMgIT0gMTA1OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjb250cm9sbGVkIFBsYW50V2lsZCBzY3JhdGNoIHByb3RvY29sIHJlcXVpcmVzIDEwNSBjbGFzc2VzIikKICAgICAgICBpZiBzZWxmLmltYWdlX3NpemUgIT0gMjI0OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjb250cm9sbGVkIFBsYW50V2lsZCBzY3JhdGNoIHByb3RvY29sIHJlcXVpcmVzIDIyNCBweCIpCiAgICAgICAgaWYgc2VsZi5tYXhfZXBvY2hzICE9IDEwMCBvciBzZWxmLnNtb2tlX2Vwb2NocyAhPSAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzY3JhdGNoIHByb3RvY29sIHJlcXVpcmVzIDEwMCBmdWxsIGFuZCAxIHNtb2tlIGVwb2NoIikKICAgICAgICBpZiBzZWxmLndhcm11cF9lcG9jaHMgIT0gNToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic2NyYXRjaCBwcm90b2NvbCByZXF1aXJlcyBmaXZlIHdhcm0tdXAgZXBvY2hzIikKICAgICAgICBpZiBzZWxmLnRyYWluX2JhdGNoX3NpemUgPD0gMCBvciBzZWxmLmV2YWxfYmF0Y2hfc2l6ZSA8PSAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJiYXRjaCBzaXplcyBtdXN0IGJlIHBvc2l0aXZlIikKICAgICAgICBpZiBzZWxmLmxlYXJuaW5nX3JhdGUgPD0gMC4wIG9yIHNlbGYud2VpZ2h0X2RlY2F5IDwgMC4wOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJvcHRpbWl6ZXIgaHlwZXJwYXJhbWV0ZXJzIGFyZSBpbnZhbGlkIikKICAgICAgICBpZiBub3QgMC4wIDw9IHNlbGYubGFiZWxfc21vb3RoaW5nIDwgMS4wOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJsYWJlbF9zbW9vdGhpbmcgbXVzdCBiZSBpbiBbMCwgMSkiKQogICAgICAgIGlmIHNlbGYuZ3JhZGllbnRfY2xpcF9ub3JtIDw9IDAuMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZ3JhZGllbnRfY2xpcF9ub3JtIG11c3QgYmUgcG9zaXRpdmUiKQogICAgICAgIGlmIHNlbGYuZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzIDwgMToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzIG11c3QgYmUgcG9zaXRpdmUiKQogICAgICAgIGlmIG5vdCAwLjAgPCBzZWxmLm1pbl9scl9yYXRpbyA8PSAxLjA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1pbl9scl9yYXRpbyBtdXN0IGJlIGluICgwLCAxXSIpCiAgICAgICAgaWYgc2VsZi5lYXJseV9zdG9wcGluZ19wYXRpZW5jZSAhPSAzMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic2NyYXRjaCBwcm90b2NvbCByZXF1aXJlcyBlYXJseS1zdG9wcGluZyBwYXRpZW5jZSAzMCIpCiAgICAgICAgaWYgc2VsZi5lYXJseV9zdG9wcGluZ19taW5fZGVsdGEgIT0gMC4wMDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNjcmF0Y2ggcHJvdG9jb2wgcmVxdWlyZXMgZWFybHktc3RvcHBpbmcgbWluX2RlbHRhIDAuMDAxIikKICAgICAgICBpZiBzZWxmLmVhcmx5X3N0b3BwaW5nX3N0YXJ0X2Vwb2NoICE9IDEwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzY3JhdGNoIHByb3RvY29sIGJlZ2lucyBlYXJseS1zdG9wcGluZyBtb25pdG9yaW5nIGF0IGVwb2NoIDEwIikKCgpkZWYgYWN0aXZlX3NlZWRzKHJ1bl9tb2RlOiBzdHIsIHNlZWRzOiBTZXF1ZW5jZVtpbnRdKSAtPiBsaXN0W2ludF06CiAgICBpZiB0dXBsZShzZWVkcykgIT0gKDQyLCAyMDI0LCAzNDA3KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzZWVkcyBtdXN0IGJlIGV4YWN0bHkgWzQyLCAyMDI0LCAzNDA3XSIpCiAgICBpZiBydW5fbW9kZSBpbiB7InNtb2tlIiwgInNjcmVlbmluZyJ9OgogICAgICAgIHJldHVybiBbNDJdCiAgICBpZiBydW5fbW9kZSA9PSAiZGV2ZWxvcG1lbnQiOgogICAgICAgIHJldHVybiBsaXN0KHNlZWRzKQogICAgcmFpc2UgVmFsdWVFcnJvcigicnVuX21vZGUgbXVzdCBiZSBzbW9rZSwgc2NyZWVuaW5nLCBvciBkZXZlbG9wbWVudCIpCgoKZGVmIGVwb2Noc19mb3JfbW9kZShjb25maWc6IFNjcmF0Y2hDb25maWcpIC0+IGludDoKICAgIHJldHVybiBjb25maWcuc21va2VfZXBvY2hzIGlmIGNvbmZpZy5ydW5fbW9kZSA9PSAic21va2UiIGVsc2UgY29uZmlnLm1heF9lcG9jaHMKCgpkZWYgc2hvdWxkX3N0ZXBfb3B0aW1pemVyKAogICAgYmF0Y2hfaW5kZXg6IGludCwKICAgICosCiAgICB0b3RhbF9iYXRjaGVzOiBpbnQsCiAgICBhY2N1bXVsYXRpb25fc3RlcHM6IGludCwKKSAtPiBib29sOgogICAgaWYgYWNjdW11bGF0aW9uX3N0ZXBzIDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJhY2N1bXVsYXRpb25fc3RlcHMgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICBpZiBiYXRjaF9pbmRleCA8IDEgb3IgYmF0Y2hfaW5kZXggPiB0b3RhbF9iYXRjaGVzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJhdGNoX2luZGV4IG11c3QgYmUgd2l0aGluIHRoZSBsb2FkZXIiKQogICAgcmV0dXJuIGJhdGNoX2luZGV4ICUgYWNjdW11bGF0aW9uX3N0ZXBzID09IDAgb3IgYmF0Y2hfaW5kZXggPT0gdG90YWxfYmF0Y2hlcwoKCmRlZiB3YXJtdXBfZXBvY2hzX2Zvcl9tb2RlKGNvbmZpZzogU2NyYXRjaENvbmZpZykgLT4gaW50OgogICAgcmV0dXJuIDAgaWYgY29uZmlnLnJ1bl9tb2RlID09ICJzbW9rZSIgZWxzZSBjb25maWcud2FybXVwX2Vwb2NocwoKCmRlZiBhc3NlcnRfbm9fdGVzdF9hY2Nlc3MocGF0aHM6IEl0ZXJhYmxlW3N0ciB8IFBhdGhdKSAtPiBOb25lOgogICAgZm9yIHZhbHVlIGluIHBhdGhzOgogICAgICAgIHBhdGggPSBQYXRoKHZhbHVlKQogICAgICAgIGZvcmJpZGRlbiA9IGFueSgKICAgICAgICAgICAgcGFydC5jYXNlZm9sZCgpLnNwbGl0KCIuIiwgMSlbMF0gPT0gInRlc3QiCiAgICAgICAgICAgIG9yIHBhcnQuY2FzZWZvbGQoKS5zdGFydHN3aXRoKCJ0ZXN0XyIpCiAgICAgICAgICAgIGZvciBwYXJ0IGluIHBhdGgucGFydHMKICAgICAgICApCiAgICAgICAgaWYgZm9yYmlkZGVuOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidGVzdCBhY2Nlc3MgaXMgZm9yYmlkZGVuIGluIGRldmVsb3BtZW50OiB7cGF0aH0iKQoKCmRlZiBjbGFzc193ZWlnaHRfdmFsdWVzKGxhYmVsX2lkczogU2VxdWVuY2VbaW50XSwgbnVtX2NsYXNzZXM6IGludCkgLT4gbGlzdFtmbG9hdF06CiAgICBpZiBudW1fY2xhc3NlcyA8PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm51bV9jbGFzc2VzIG11c3QgYmUgcG9zaXRpdmUiKQogICAgY291bnRzID0gWzBdICogbnVtX2NsYXNzZXMKICAgIGZvciB2YWx1ZSBpbiBsYWJlbF9pZHM6CiAgICAgICAgbGFiZWwgPSBpbnQodmFsdWUpCiAgICAgICAgaWYgbGFiZWwgbm90IGluIHJhbmdlKG51bV9jbGFzc2VzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVua25vd24gdHJhaW5pbmcgbGFiZWw6IHtsYWJlbH0iKQogICAgICAgIGNvdW50c1tsYWJlbF0gKz0gMQogICAgaWYgYW55KGNvdW50IDw9IDAgZm9yIGNvdW50IGluIGNvdW50cyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZWFjaCBjbGFzcyByZXF1aXJlcyB0cmFpbmluZyBzdXBwb3J0IikKICAgIHNhbXBsZV9jb3VudCA9IHN1bShjb3VudHMpCiAgICByYXcgPSBbbWF0aC5zcXJ0KHNhbXBsZV9jb3VudCAvIGNvdW50KSBmb3IgY291bnQgaW4gY291bnRzXQogICAgbWVhbiA9IHN1bShyYXcpIC8gbGVuKHJhdykKICAgIHJldHVybiBbdmFsdWUgLyBtZWFuIGZvciB2YWx1ZSBpbiByYXddCgoKQGRhdGFjbGFzcwpjbGFzcyBFYXJseVN0b3BwaW5nU3RhdGU6CiAgICBwYXRpZW5jZTogaW50CiAgICBtaW5fZGVsdGE6IGZsb2F0CiAgICBzdGFydF9lcG9jaDogaW50CiAgICBiZXN0X21hY3JvX2YxOiBmbG9hdCB8IE5vbmUgPSBOb25lCiAgICBiZXN0X2Vwb2NoOiBpbnQgfCBOb25lID0gTm9uZQogICAgYmFkX2Vwb2NoczogaW50ID0gMAogICAgc3RvcF9lcG9jaDogaW50IHwgTm9uZSA9IE5vbmUKCiAgICBkZWYgX19wb3N0X2luaXRfXyhzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIHNlbGYucGF0aWVuY2UgPCAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwYXRpZW5jZSBtdXN0IGJlIHBvc2l0aXZlIikKICAgICAgICBpZiBzZWxmLm1pbl9kZWx0YSA8IDAuMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibWluX2RlbHRhIG11c3QgYmUgbm9ubmVnYXRpdmUiKQogICAgICAgIGlmIHNlbGYuc3RhcnRfZXBvY2ggPCAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzdGFydF9lcG9jaCBtdXN0IGJlIHBvc2l0aXZlIikKCiAgICBkZWYgdXBkYXRlKHNlbGYsIGVwb2NoOiBpbnQsIG1hY3JvX2YxOiBmbG9hdCkgLT4gYm9vbDoKICAgICAgICBpZiBzZWxmLnN0b3BfZXBvY2ggaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAgICAgaWYgZXBvY2ggPCBzZWxmLnN0YXJ0X2Vwb2NoOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICBzY29yZSA9IGZsb2F0KG1hY3JvX2YxKQogICAgICAgIGlmIG5vdCBtYXRoLmlzZmluaXRlKHNjb3JlKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZWFybHktc3RvcHBpbmcgc2NvcmUgbXVzdCBiZSBmaW5pdGUiKQogICAgICAgIGlmIHNlbGYuYmVzdF9tYWNyb19mMSBpcyBOb25lIG9yIHNjb3JlID4gc2VsZi5iZXN0X21hY3JvX2YxICsgc2VsZi5taW5fZGVsdGE6CiAgICAgICAgICAgIHNlbGYuYmVzdF9tYWNyb19mMSA9IHNjb3JlCiAgICAgICAgICAgIHNlbGYuYmVzdF9lcG9jaCA9IGludChlcG9jaCkKICAgICAgICAgICAgc2VsZi5iYWRfZXBvY2hzID0gMAogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICBzZWxmLmJhZF9lcG9jaHMgKz0gMQogICAgICAgIGlmIHNlbGYuYmFkX2Vwb2NocyA+PSBzZWxmLnBhdGllbmNlOgogICAgICAgICAgICBzZWxmLnN0b3BfZXBvY2ggPSBpbnQoZXBvY2gpCiAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAgICAgcmV0dXJuIEZhbHNlCgogICAgZGVmIGFzX2RpY3Qoc2VsZikgLT4gZGljdFtzdHIsIEFueV06CiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgInBhdGllbmNlIjogc2VsZi5wYXRpZW5jZSwKICAgICAgICAgICAgIm1pbl9kZWx0YSI6IHNlbGYubWluX2RlbHRhLAogICAgICAgICAgICAic3RhcnRfZXBvY2giOiBzZWxmLnN0YXJ0X2Vwb2NoLAogICAgICAgICAgICAiYmVzdF9tYWNyb19mMSI6IHNlbGYuYmVzdF9tYWNyb19mMSwKICAgICAgICAgICAgImJlc3RfZXBvY2giOiBzZWxmLmJlc3RfZXBvY2gsCiAgICAgICAgICAgICJiYWRfZXBvY2hzIjogc2VsZi5iYWRfZXBvY2hzLAogICAgICAgICAgICAic3RvcF9lcG9jaCI6IHNlbGYuc3RvcF9lcG9jaCwKICAgICAgICB9CgoKY2xhc3MgV2FybXVwQ29zaW5lU2NoZWR1bGVyOgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgb3B0aW1pemVyOiBBbnksCiAgICAgICAgKiwKICAgICAgICB0b3RhbF9lcG9jaHM6IGludCwKICAgICAgICB3YXJtdXBfZXBvY2hzOiBpbnQsCiAgICAgICAgbWluX2xyX3JhdGlvOiBmbG9hdCwKICAgICkgLT4gTm9uZToKICAgICAgICBpZiB0b3RhbF9lcG9jaHMgPCAxIG9yIHdhcm11cF9lcG9jaHMgPCAwIG9yIHdhcm11cF9lcG9jaHMgPj0gdG90YWxfZXBvY2hzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzY2hlZHVsZXIgZXBvY2ggY29uZmlndXJhdGlvbiBpcyBpbnZhbGlkIikKICAgICAgICBpZiBub3QgMC4wIDwgbWluX2xyX3JhdGlvIDw9IDEuMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibWluX2xyX3JhdGlvIG11c3QgYmUgaW4gKDAsIDFdIikKICAgICAgICBzZWxmLm9wdGltaXplciA9IG9wdGltaXplcgogICAgICAgIHNlbGYudG90YWxfZXBvY2hzID0gdG90YWxfZXBvY2hzCiAgICAgICAgc2VsZi53YXJtdXBfZXBvY2hzID0gd2FybXVwX2Vwb2NocwogICAgICAgIHNlbGYubWluX2xyX3JhdGlvID0gbWluX2xyX3JhdGlvCiAgICAgICAgc2VsZi5iYXNlX2xycyA9IFtmbG9hdChncm91cFsibHIiXSkgZm9yIGdyb3VwIGluIG9wdGltaXplci5wYXJhbV9ncm91cHNdCgogICAgZGVmIF9zY2FsZShzZWxmLCBlcG9jaDogaW50KSAtPiBmbG9hdDoKICAgICAgICBpZiBlcG9jaCBub3QgaW4gcmFuZ2UoMSwgc2VsZi50b3RhbF9lcG9jaHMgKyAxKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZXBvY2ggaXMgb3V0c2lkZSB0aGUgc2NoZWR1bGVyIGJ1ZGdldCIpCiAgICAgICAgaWYgc2VsZi53YXJtdXBfZXBvY2hzIGFuZCBlcG9jaCA8PSBzZWxmLndhcm11cF9lcG9jaHM6CiAgICAgICAgICAgIHJldHVybiBlcG9jaCAvIHNlbGYud2FybXVwX2Vwb2NocwogICAgICAgIGRlY2F5X2Vwb2NocyA9IHNlbGYudG90YWxfZXBvY2hzIC0gc2VsZi53YXJtdXBfZXBvY2hzCiAgICAgICAgZGVjYXlfaW5kZXggPSBlcG9jaCAtIHNlbGYud2FybXVwX2Vwb2NocyAtIDEKICAgICAgICBwcm9ncmVzcyA9IGRlY2F5X2luZGV4IC8gbWF4KDEsIGRlY2F5X2Vwb2NocyAtIDEpCiAgICAgICAgY29zaW5lID0gMC41ICogKDEuMCArIG1hdGguY29zKG1hdGgucGkgKiBwcm9ncmVzcykpCiAgICAgICAgcmV0dXJuIHNlbGYubWluX2xyX3JhdGlvICsgKDEuMCAtIHNlbGYubWluX2xyX3JhdGlvKSAqIGNvc2luZQoKICAgIGRlZiBzdGVwKHNlbGYsIGVwb2NoOiBpbnQpIC0+IGxpc3RbZmxvYXRdOgogICAgICAgIHNjYWxlID0gc2VsZi5fc2NhbGUoZXBvY2gpCiAgICAgICAgdmFsdWVzID0gW2Jhc2VfbHIgKiBzY2FsZSBmb3IgYmFzZV9sciBpbiBzZWxmLmJhc2VfbHJzXQogICAgICAgIGZvciBncm91cCwgdmFsdWUgaW4gemlwKHNlbGYub3B0aW1pemVyLnBhcmFtX2dyb3VwcywgdmFsdWVzKToKICAgICAgICAgICAgZ3JvdXBbImxyIl0gPSB2YWx1ZQogICAgICAgIHJldHVybiB2YWx1ZXMKCgpkZWYgX3BhcmFtZXRlcl9tb2R1bGVzKG1vZGVsOiBBbnkpIC0+IGRpY3RbaW50LCB0dXBsZVtzdHIsIEFueSwgc3RyXV06CiAgICBvd25lcnM6IGRpY3RbaW50LCB0dXBsZVtzdHIsIEFueSwgc3RyXV0gPSB7fQogICAgZm9yIG1vZHVsZV9wYXRoLCBtb2R1bGUgaW4gbW9kZWwubmFtZWRfbW9kdWxlcygpOgogICAgICAgIGZvciBsb2NhbF9uYW1lLCBwYXJhbWV0ZXIgaW4gbW9kdWxlLm5hbWVkX3BhcmFtZXRlcnMocmVjdXJzZT1GYWxzZSk6CiAgICAgICAgICAgIGZ1bGxfbmFtZSA9IGYie21vZHVsZV9wYXRofS57bG9jYWxfbmFtZX0iIGlmIG1vZHVsZV9wYXRoIGVsc2UgbG9jYWxfbmFtZQogICAgICAgICAgICBvd25lcnMuc2V0ZGVmYXVsdChpZChwYXJhbWV0ZXIpLCAoZnVsbF9uYW1lLCBtb2R1bGUsIGxvY2FsX25hbWUpKQogICAgcmV0dXJuIG93bmVycwoKCmRlZiBfdXNlc196ZXJvX2RlY2F5KG1vZHVsZTogQW55LCBsb2NhbF9uYW1lOiBzdHIpIC0+IGJvb2w6CiAgICBpbXBvcnQgdG9yY2gubm4gYXMgbm4KCiAgICBub3JtYWxpemF0aW9uX3R5cGVzID0gKAogICAgICAgIG5uLm1vZHVsZXMuYmF0Y2hub3JtLl9CYXRjaE5vcm0sCiAgICAgICAgbm4uTGF5ZXJOb3JtLAogICAgICAgIG5uLkdyb3VwTm9ybSwKICAgICAgICBubi5tb2R1bGVzLmluc3RhbmNlbm9ybS5fSW5zdGFuY2VOb3JtLAogICAgKQogICAgY2xhc3NfbmFtZSA9IG1vZHVsZS5fX2NsYXNzX18uX19uYW1lX18uY2FzZWZvbGQoKQogICAgcmV0dXJuICgKICAgICAgICBsb2NhbF9uYW1lLmNhc2Vmb2xkKCkgaW4geyJiaWFzIiwgImdhbW1hIiwgImxheWVyX3NjYWxlIiwgImxheWVyc2NhbGUifQogICAgICAgIG9yIGlzaW5zdGFuY2UobW9kdWxlLCBub3JtYWxpemF0aW9uX3R5cGVzKQogICAgICAgIG9yICJub3JtIiBpbiBjbGFzc19uYW1lCiAgICAgICAgb3IgImxheWVyc2NhbGUiIGluIGNsYXNzX25hbWUKICAgICkKCgpkZWYgYnVpbGRfb3B0aW1pemVyX2dyb3VwcygKICAgIG1vZGVsOiBBbnksCiAgICAqLAogICAgbGVhcm5pbmdfcmF0ZTogZmxvYXQsCiAgICB3ZWlnaHRfZGVjYXk6IGZsb2F0LAopIC0+IHR1cGxlW2xpc3RbZGljdFtzdHIsIEFueV1dLCBsaXN0W2RpY3Rbc3RyLCBBbnldXV06CiAgICB0cmFpbmFibGUgPSBbcGFyYW1ldGVyIGZvciBwYXJhbWV0ZXIgaW4gbW9kZWwucGFyYW1ldGVycygpIGlmIHBhcmFtZXRlci5yZXF1aXJlc19ncmFkXQogICAgb3duZXJzID0gX3BhcmFtZXRlcl9tb2R1bGVzKG1vZGVsKQogICAgYnVja2V0czogZGljdFtzdHIsIGxpc3RbQW55XV0gPSB7ImRlY2F5IjogW10sICJub19kZWNheSI6IFtdfQogICAgYXVkaXQgPSBbXQogICAgZm9yIHBhcmFtZXRlciBpbiB0cmFpbmFibGU6CiAgICAgICAgaWYgaWQocGFyYW1ldGVyKSBub3QgaW4gb3duZXJzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0cmFpbmFibGUgcGFyYW1ldGVyIGhhcyBubyBvd25pbmcgbW9kdWxlIikKICAgICAgICBwYXJhbWV0ZXJfbmFtZSwgbW9kdWxlLCBsb2NhbF9uYW1lID0gb3duZXJzW2lkKHBhcmFtZXRlcildCiAgICAgICAgYnVja2V0ID0gIm5vX2RlY2F5IiBpZiBfdXNlc196ZXJvX2RlY2F5KG1vZHVsZSwgbG9jYWxfbmFtZSkgZWxzZSAiZGVjYXkiCiAgICAgICAgYXBwbGllZF9kZWNheSA9IDAuMCBpZiBidWNrZXQgPT0gIm5vX2RlY2F5IiBlbHNlIHdlaWdodF9kZWNheQogICAgICAgIGJ1Y2tldHNbYnVja2V0XS5hcHBlbmQocGFyYW1ldGVyKQogICAgICAgIGF1ZGl0LmFwcGVuZCgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgInBhcmFtZXRlcl9uYW1lIjogcGFyYW1ldGVyX25hbWUsCiAgICAgICAgICAgICAgICAiZ3JvdXAiOiBidWNrZXQsCiAgICAgICAgICAgICAgICAibHIiOiBsZWFybmluZ19yYXRlLAogICAgICAgICAgICAgICAgIndlaWdodF9kZWNheSI6IGFwcGxpZWRfZGVjYXksCiAgICAgICAgICAgICAgICAibnVtZWwiOiBwYXJhbWV0ZXIubnVtZWwoKSwKICAgICAgICAgICAgfQogICAgICAgICkKICAgIGdyb3VwZWQgPSBidWNrZXRzWyJkZWNheSJdICsgYnVja2V0c1sibm9fZGVjYXkiXQogICAgaWYgbGVuKHtpZChwYXJhbWV0ZXIpIGZvciBwYXJhbWV0ZXIgaW4gZ3JvdXBlZH0pICE9IGxlbihncm91cGVkKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJvcHRpbWl6ZXIgcGFyYW1ldGVyIGdyb3VwcyBvdmVybGFwIikKICAgIGlmIHtpZChwYXJhbWV0ZXIpIGZvciBwYXJhbWV0ZXIgaW4gZ3JvdXBlZH0gIT0ge2lkKHBhcmFtZXRlcikgZm9yIHBhcmFtZXRlciBpbiB0cmFpbmFibGV9OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm9wdGltaXplciBwYXJhbWV0ZXIgZ3JvdXBzIGRvIG5vdCBjb3ZlciB0aGUgbW9kZWwiKQogICAgZ3JvdXBzID0gWwogICAgICAgIHsKICAgICAgICAgICAgIm5hbWUiOiBuYW1lLAogICAgICAgICAgICAicGFyYW1zIjogcGFyYW1ldGVycywKICAgICAgICAgICAgImxyIjogbGVhcm5pbmdfcmF0ZSwKICAgICAgICAgICAgIndlaWdodF9kZWNheSI6IDAuMCBpZiBuYW1lID09ICJub19kZWNheSIgZWxzZSB3ZWlnaHRfZGVjYXksCiAgICAgICAgfQogICAgICAgIGZvciBuYW1lLCBwYXJhbWV0ZXJzIGluIGJ1Y2tldHMuaXRlbXMoKQogICAgXQogICAgcmV0dXJuIGdyb3Vwcywgc29ydGVkKGF1ZGl0LCBrZXk9bGFtYmRhIHJvdzogcm93WyJwYXJhbWV0ZXJfbmFtZSJdKQoKCmRlZiBjaGVja3BvaW50X2lzX2JldHRlcigKICAgIGNhbmRpZGF0ZTogZGljdFtzdHIsIGZsb2F0IHwgaW50XSwKICAgIGJlc3Q6IGRpY3Rbc3RyLCBmbG9hdCB8IGludF0gfCBOb25lLAopIC0+IGJvb2w6CiAgICBpZiBiZXN0IGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIFRydWUKICAgIGNhbmRpZGF0ZV9mMSA9IGZsb2F0KGNhbmRpZGF0ZVsibWFjcm9fZjEiXSkKICAgIGJlc3RfZjEgPSBmbG9hdChiZXN0WyJtYWNyb19mMSJdKQogICAgaWYgY2FuZGlkYXRlX2YxID4gYmVzdF9mMSArIDFlLTEyOgogICAgICAgIHJldHVybiBUcnVlCiAgICBpZiBhYnMoY2FuZGlkYXRlX2YxIC0gYmVzdF9mMSkgPD0gMWUtMTI6CiAgICAgICAgY2FuZGlkYXRlX25sbCA9IGZsb2F0KGNhbmRpZGF0ZVsibmxsIl0pCiAgICAgICAgYmVzdF9ubGwgPSBmbG9hdChiZXN0WyJubGwiXSkKICAgICAgICBpZiBjYW5kaWRhdGVfbmxsIDwgYmVzdF9ubGwgLSAxZS0xMjoKICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICBpZiBhYnMoY2FuZGlkYXRlX25sbCAtIGJlc3RfbmxsKSA8PSAxZS0xMjoKICAgICAgICAgICAgcmV0dXJuIGludChjYW5kaWRhdGVbImVwb2NoIl0pIDwgaW50KGJlc3RbImVwb2NoIl0pCiAgICByZXR1cm4gRmFsc2UK')
EMBEDDED_RUNTIME_SOURCE_BYTES = base64.b64decode('...')


In [4]:
"""Pure contracts for the controlled PlantWild scratch comparison."""

from __future__ import annotations

import math
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Sequence


@dataclass(frozen=True)
class ModelSpec:
    key: str
    display_name: str
    kind: str
    timm_name: str | None
    expected_parameters: int
    pretrained: bool = False


MODEL_SPECS = {
    "wavestarnet_big_transfer_v2": ModelSpec(
        key="wavestarnet_big_transfer_v2",
        display_name="WavestarNet-Big-Transfer-V2",
        kind="wavestarnet",
        timm_name=None,
        expected_parameters=3_013_109,
    ),
    "mobilenetv3_large_100": ModelSpec(
        key="mobilenetv3_large_100",
        display_name="MobileNetV3-Large",
        kind="timm",
        timm_name="mobilenetv3_large_100",
        expected_parameters=4_336_537,
    ),
    "efficientnet_b0": ModelSpec(
        key="efficientnet_b0",
        display_name="EfficientNet-B0",
        kind="timm",
        timm_name="efficientnet_b0",
        expected_parameters=4_142_053,
    ),
}


def model_spec(key: str) -> ModelSpec:
    try:
        return MODEL_SPECS[key]
    except KeyError as error:
        raise ValueError(f"unknown scratch model_key: {key}") from error


@dataclass(frozen=True)
class ScratchConfig:
    model_key: str
    run_mode: str
    seeds: tuple[int, ...] = (42, 2024, 3407)
    num_classes: int = 105
    image_size: int = 224
    train_batch_size: int = 32
    eval_batch_size: int = 64
    max_epochs: int = 100
    smoke_epochs: int = 1
    warmup_epochs: int = 5
    learning_rate: float = 3e-4
    weight_decay: float = 0.05
    label_smoothing: float = 0.05
    gradient_clip_norm: float = 1.0
    gradient_accumulation_steps: int = 1
    min_lr_ratio: float = 0.05
    early_stopping_patience: int = 30
    early_stopping_min_delta: float = 0.001
    early_stopping_start_epoch: int = 10

    def __post_init__(self) -> None:
        model_spec(self.model_key)
        active_seeds(self.run_mode, self.seeds)
        if self.num_classes != 105:
            raise ValueError("controlled PlantWild scratch protocol requires 105 classes")
        if self.image_size != 224:
            raise ValueError("controlled PlantWild scratch protocol requires 224 px")
        if self.max_epochs != 100 or self.smoke_epochs != 1:
            raise ValueError("scratch protocol requires 100 full and 1 smoke epoch")
        if self.warmup_epochs != 5:
            raise ValueError("scratch protocol requires five warm-up epochs")
        if self.train_batch_size <= 0 or self.eval_batch_size <= 0:
            raise ValueError("batch sizes must be positive")
        if self.learning_rate <= 0.0 or self.weight_decay < 0.0:
            raise ValueError("optimizer hyperparameters are invalid")
        if not 0.0 <= self.label_smoothing < 1.0:
            raise ValueError("label_smoothing must be in [0, 1)")
        if self.gradient_clip_norm <= 0.0:
            raise ValueError("gradient_clip_norm must be positive")
        if self.gradient_accumulation_steps < 1:
            raise ValueError("gradient_accumulation_steps must be positive")
        if not 0.0 < self.min_lr_ratio <= 1.0:
            raise ValueError("min_lr_ratio must be in (0, 1]")
        if self.early_stopping_patience != 30:
            raise ValueError("scratch protocol requires early-stopping patience 30")
        if self.early_stopping_min_delta != 0.001:
            raise ValueError("scratch protocol requires early-stopping min_delta 0.001")
        if self.early_stopping_start_epoch != 10:
            raise ValueError("scratch protocol begins early-stopping monitoring at epoch 10")


def active_seeds(run_mode: str, seeds: Sequence[int]) -> list[int]:
    if tuple(seeds) != (42, 2024, 3407):
        raise ValueError("seeds must be exactly [42, 2024, 3407]")
    if run_mode in {"smoke", "screening"}:
        return [42]
    if run_mode == "development":
        return list(seeds)
    raise ValueError("run_mode must be smoke, screening, or development")


def epochs_for_mode(config: ScratchConfig) -> int:
    return config.smoke_epochs if config.run_mode == "smoke" else config.max_epochs


def should_step_optimizer(
    batch_index: int,
    *,
    total_batches: int,
    accumulation_steps: int,
) -> bool:
    if accumulation_steps < 1:
        raise ValueError("accumulation_steps must be positive")
    if batch_index < 1 or batch_index > total_batches:
        raise ValueError("batch_index must be within the loader")
    return batch_index % accumulation_steps == 0 or batch_index == total_batches


def warmup_epochs_for_mode(config: ScratchConfig) -> int:
    return 0 if config.run_mode == "smoke" else config.warmup_epochs


def assert_no_test_access(paths: Iterable[str | Path]) -> None:
    for value in paths:
        path = Path(value)
        forbidden = any(
            part.casefold().split(".", 1)[0] == "test"
            or part.casefold().startswith("test_")
            for part in path.parts
        )
        if forbidden:
            raise ValueError(f"test access is forbidden in development: {path}")


def class_weight_values(label_ids: Sequence[int], num_classes: int) -> list[float]:
    if num_classes <= 0:
        raise ValueError("num_classes must be positive")
    counts = [0] * num_classes
    for value in label_ids:
        label = int(value)
        if label not in range(num_classes):
            raise ValueError(f"unknown training label: {label}")
        counts[label] += 1
    if any(count <= 0 for count in counts):
        raise ValueError("each class requires training support")
    sample_count = sum(counts)
    raw = [math.sqrt(sample_count / count) for count in counts]
    mean = sum(raw) / len(raw)
    return [value / mean for value in raw]


@dataclass
class EarlyStoppingState:
    patience: int
    min_delta: float
    start_epoch: int
    best_macro_f1: float | None = None
    best_epoch: int | None = None
    bad_epochs: int = 0
    stop_epoch: int | None = None

    def __post_init__(self) -> None:
        if self.patience < 1:
            raise ValueError("patience must be positive")
        if self.min_delta < 0.0:
            raise ValueError("min_delta must be nonnegative")
        if self.start_epoch < 1:
            raise ValueError("start_epoch must be positive")

    def update(self, epoch: int, macro_f1: float) -> bool:
        if self.stop_epoch is not None:
            return True
        if epoch < self.start_epoch:
            return False
        score = float(macro_f1)
        if not math.isfinite(score):
            raise ValueError("early-stopping score must be finite")
        if self.best_macro_f1 is None or score > self.best_macro_f1 + self.min_delta:
            self.best_macro_f1 = score
            self.best_epoch = int(epoch)
            self.bad_epochs = 0
            return False
        self.bad_epochs += 1
        if self.bad_epochs >= self.patience:
            self.stop_epoch = int(epoch)
            return True
        return False

    def as_dict(self) -> dict[str, Any]:
        return {
            "patience": self.patience,
            "min_delta": self.min_delta,
            "start_epoch": self.start_epoch,
            "best_macro_f1": self.best_macro_f1,
            "best_epoch": self.best_epoch,
            "bad_epochs": self.bad_epochs,
            "stop_epoch": self.stop_epoch,
        }


class WarmupCosineScheduler:
    def __init__(
        self,
        optimizer: Any,
        *,
        total_epochs: int,
        warmup_epochs: int,
        min_lr_ratio: float,
    ) -> None:
        if total_epochs < 1 or warmup_epochs < 0 or warmup_epochs >= total_epochs:
            raise ValueError("scheduler epoch configuration is invalid")
        if not 0.0 < min_lr_ratio <= 1.0:
            raise ValueError("min_lr_ratio must be in (0, 1]")
        self.optimizer = optimizer
        self.total_epochs = total_epochs
        self.warmup_epochs = warmup_epochs
        self.min_lr_ratio = min_lr_ratio
        self.base_lrs = [float(group["lr"]) for group in optimizer.param_groups]

    def _scale(self, epoch: int) -> float:
        if epoch not in range(1, self.total_epochs + 1):
            raise ValueError("epoch is outside the scheduler budget")
        if self.warmup_epochs and epoch <= self.warmup_epochs:
            return epoch / self.warmup_epochs
        decay_epochs = self.total_epochs - self.warmup_epochs
        decay_index = epoch - self.warmup_epochs - 1
        progress = decay_index / max(1, decay_epochs - 1)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return self.min_lr_ratio + (1.0 - self.min_lr_ratio) * cosine

    def step(self, epoch: int) -> list[float]:
        scale = self._scale(epoch)
        values = [base_lr * scale for base_lr in self.base_lrs]
        for group, value in zip(self.optimizer.param_groups, values):
            group["lr"] = value
        return values


def _parameter_modules(model: Any) -> dict[int, tuple[str, Any, str]]:
    owners: dict[int, tuple[str, Any, str]] = {}
    for module_path, module in model.named_modules():
        for local_name, parameter in module.named_parameters(recurse=False):
            full_name = f"{module_path}.{local_name}" if module_path else local_name
            owners.setdefault(id(parameter), (full_name, module, local_name))
    return owners


def _uses_zero_decay(module: Any, local_name: str) -> bool:
    import torch.nn as nn

    normalization_types = (
        nn.modules.batchnorm._BatchNorm,
        nn.LayerNorm,
        nn.GroupNorm,
        nn.modules.instancenorm._InstanceNorm,
    )
    class_name = module.__class__.__name__.casefold()
    return (
        local_name.casefold() in {"bias", "gamma", "layer_scale", "layerscale"}
        or isinstance(module, normalization_types)
        or "norm" in class_name
        or "layerscale" in class_name
    )


def build_optimizer_groups(
    model: Any,
    *,
    learning_rate: float,
    weight_decay: float,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    owners = _parameter_modules(model)
    buckets: dict[str, list[Any]] = {"decay": [], "no_decay": []}
    audit = []
    for parameter in trainable:
        if id(parameter) not in owners:
            raise ValueError("trainable parameter has no owning module")
        parameter_name, module, local_name = owners[id(parameter)]
        bucket = "no_decay" if _uses_zero_decay(module, local_name) else "decay"
        applied_decay = 0.0 if bucket == "no_decay" else weight_decay
        buckets[bucket].append(parameter)
        audit.append(
            {
                "parameter_name": parameter_name,
                "group": bucket,
                "lr": learning_rate,
                "weight_decay": applied_decay,
                "numel": parameter.numel(),
            }
        )
    grouped = buckets["decay"] + buckets["no_decay"]
    if len({id(parameter) for parameter in grouped}) != len(grouped):
        raise ValueError("optimizer parameter groups overlap")
    if {id(parameter) for parameter in grouped} != {id(parameter) for parameter in trainable}:
        raise ValueError("optimizer parameter groups do not cover the model")
    groups = [
        {
            "name": name,
            "params": parameters,
            "lr": learning_rate,
            "weight_decay": 0.0 if name == "no_decay" else weight_decay,
        }
        for name, parameters in buckets.items()
    ]
    return groups, sorted(audit, key=lambda row: row["parameter_name"])


def checkpoint_is_better(
    candidate: dict[str, float | int],
    best: dict[str, float | int] | None,
) -> bool:
    if best is None:
        return True
    candidate_f1 = float(candidate["macro_f1"])
    best_f1 = float(best["macro_f1"])
    if candidate_f1 > best_f1 + 1e-12:
        return True
    if abs(candidate_f1 - best_f1) <= 1e-12:
        candidate_nll = float(candidate["nll"])
        best_nll = float(best["nll"])
        if candidate_nll < best_nll - 1e-12:
            return True
        if abs(candidate_nll - best_nll) <= 1e-12:
            return int(candidate["epoch"]) < int(best["epoch"])
    return False


In [5]:
"""Self-contained runtime for the controlled PlantWild scratch comparison."""

from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import platform
import random
import shutil
import statistics
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as functional
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

try:
    from plantwild.scratch_comparison.experiment_core import (
        EarlyStoppingState,
        ScratchConfig,
        WarmupCosineScheduler,
        active_seeds,
        assert_no_test_access,
        build_optimizer_groups,
        checkpoint_is_better,
        class_weight_values,
        epochs_for_mode,
        model_spec,
        warmup_epochs_for_mode,
    )
except ModuleNotFoundError:
    # Generated notebooks define these symbols in the preceding core cell.
    pass


DATASET_ID = "sumitnath8987/plantwild-clean-training-v1"
NUM_CLASSES = 105
PROTOCOL_ID = "plantwild_scratch_comparison_v1"
EXPECTED_WAVESTAR_ARCHITECTURE_SHA256 = (
    "9ecf1f23cfdfb2c710ba89cc9f1620337ef96cbdc11f48a7c8c305df8798f474"
)
EXPECTED_WAVESTAR_BACKBONE_SHA256 = (
    "f642f9b1eb6ca21454566ae81d7bc76782bed77e9210b4c4edaa32c63596bf94"
)
METRIC_NAMES = (
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "macro_precision",
    "macro_recall",
    "weighted_precision",
    "weighted_recall",
    "nll",
    "brier",
    "ece_15bin",
)


@dataclass(frozen=True)
class DevelopmentBundle:
    frames: dict[str, Any]
    class_names: tuple[str, ...]
    class_to_id: dict[str, int]
    root: Path
    image_root: Path


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def write_json(payload: Any, path: str | Path) -> None:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(payload, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.set_float32_matmul_precision("high")


def seed_worker(worker_id: int) -> None:
    del worker_id
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def build_transforms() -> tuple[Any, Any]:
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)
    training = transforms.Compose(
        [
            transforms.RandomResizedCrop(
                224,
                scale=(0.70, 1.00),
                ratio=(0.80, 1.25),
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.2),
            transforms.RandomRotation(20),
            transforms.ColorJitter(0.20, 0.20, 0.15, 0.03),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )
    evaluation = transforms.Compose(
        [
            transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )
    return training, evaluation


class ManifestImageDataset(Dataset):
    def __init__(self, frame: Any, transform: Any):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        with Image.open(row.absolute_path) as image:
            tensor = self.transform(image.convert("RGB"))
        return (
            tensor,
            int(row.label_id),
            str(row.sample_id),
            str(row.component_id),
            str(row.packaged_path),
        )


def resolve_clean_dataset(input_root: str | Path) -> Path:
    root = Path(input_root).resolve()
    required = Path("manifests/train.csv")
    candidates = [root] if (root / required).is_file() else []
    candidates.extend(
        path.parent.parent
        for path in root.rglob("manifests/class_mapping.json")
        if (path.parent.parent / required).is_file()
    )
    unique = sorted(set(path.resolve() for path in candidates))
    if len(unique) != 1:
        raise ValueError(f"expected one clean dataset root, found {len(unique)}")
    return unique[0]


def load_development_bundle(
    input_root: str | Path,
    *,
    expected_hashes: dict[str, str],
    validate_images: bool = True,
) -> DevelopmentBundle:
    root = resolve_clean_dataset(input_root)
    manifests = root / "manifests"
    paths = {
        "train": manifests / "train.csv",
        "validation": manifests / "validation.csv",
        "class_mapping": manifests / "class_mapping.json",
    }
    assert_no_test_access(paths.values())
    actual_hashes = {name: sha256_file(path) for name, path in paths.items()}
    if actual_hashes != expected_hashes:
        raise ValueError(
            f"fixed manifest hash mismatch: {actual_hashes} != {expected_hashes}"
        )
    direct_image_root = (root / "plantwild-clean-grouped-images").resolve()
    image_candidates = [direct_image_root] if direct_image_root.is_dir() else []
    if validate_images and not image_candidates:
        image_candidates = sorted(
            path.resolve()
            for path in Path(input_root).rglob("plantwild-clean-grouped-images")
            if path.is_dir()
        )
    if validate_images and len(image_candidates) != 1:
        raise ValueError(f"expected one clean image root, found {len(image_candidates)}")
    image_root = image_candidates[0] if image_candidates else direct_image_root
    mapping = json.loads(paths["class_mapping"].read_text(encoding="utf-8"))
    class_to_id = {
        str(name): int(label_id)
        for name, label_id in mapping["class_to_id"].items()
    }
    id_to_class = {
        int(label_id): str(name)
        for label_id, name in mapping["id_to_class"].items()
    }
    if sorted(class_to_id.values()) != list(range(NUM_CLASSES)):
        raise ValueError("class IDs must be contiguous range(105)")
    class_names = tuple(id_to_class[index] for index in range(NUM_CLASSES))
    frames: dict[str, Any] = {}
    sample_ids: set[str] = set()
    component_owner: dict[str, str] = {}
    required_columns = {
        "sample_id",
        "component_id",
        "split",
        "label",
        "label_id",
        "packaged_path",
    }
    for split in ("train", "validation"):
        frame = pd.read_csv(paths[split])
        missing = required_columns - set(frame.columns)
        if missing:
            raise ValueError(f"{split} manifest missing columns: {sorted(missing)}")
        if frame.empty or frame["sample_id"].duplicated().any():
            raise ValueError(f"{split} manifest is empty or has duplicate sample IDs")
        current_ids = set(frame["sample_id"].astype(str))
        if sample_ids.intersection(current_ids):
            raise ValueError("sample IDs overlap across development splits")
        sample_ids.update(current_ids)
        absolute_paths = []
        for row in frame.itertuples(index=False):
            if str(row.split) != split:
                raise ValueError(f"manifest row has wrong split: {row.sample_id}")
            if class_to_id.get(str(row.label)) != int(row.label_id):
                raise ValueError(f"class mapping mismatch: {row.sample_id}")
            component = str(row.component_id)
            owner = component_owner.get(component)
            if owner is not None and owner != split:
                raise ValueError("component IDs overlap across development splits")
            component_owner[component] = split
            candidate = (image_root / str(row.packaged_path)).resolve()
            try:
                candidate.relative_to(image_root)
            except ValueError as error:
                raise ValueError("packaged path escapes image root") from error
            if validate_images and not candidate.is_file():
                raise ValueError(f"packaged image is missing: {row.packaged_path}")
            absolute_paths.append(str(candidate))
        frame = frame.copy()
        frame["absolute_path"] = absolute_paths
        frames[split] = frame.sort_values("sample_id").reset_index(drop=True)
    for split, frame in frames.items():
        support = set(int(value) for value in frame["label_id"])
        if support != set(range(NUM_CLASSES)):
            raise ValueError(f"{split} does not contain all 105 classes")
    return DevelopmentBundle(frames, class_names, class_to_id, root, image_root)


def stratified_subset(frame: Any, per_class: int, seed: int) -> Any:
    if per_class < 1:
        raise ValueError("per_class must be positive")
    groups = [
        group.sample(n=min(per_class, len(group)), random_state=seed)
        for _, group in frame.groupby("label_id", sort=True)
    ]
    return (
        pd.concat(groups, ignore_index=True)
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )


def build_loaders(
    frames: dict[str, Any],
    seed: int,
    config: ScratchConfig,
) -> dict[str, DataLoader]:
    training_transform, evaluation_transform = build_transforms()
    generator = torch.Generator().manual_seed(seed)
    use_cuda = torch.cuda.is_available()
    common = {
        "num_workers": 2,
        "pin_memory": use_cuda,
        "persistent_workers": True,
        "worker_init_fn": seed_worker,
    }
    return {
        "train": DataLoader(
            ManifestImageDataset(frames["train"], training_transform),
            batch_size=config.train_batch_size,
            shuffle=True,
            drop_last=False,
            generator=generator,
            **common,
        ),
        "validation": DataLoader(
            ManifestImageDataset(frames["validation"], evaluation_transform),
            batch_size=config.eval_batch_size,
            shuffle=False,
            drop_last=False,
            **common,
        ),
    }


def _wavestar_symbols() -> tuple[Any, Any, Any]:
    if all(
        name in globals()
        for name in (
            "create_wavestarnet",
            "architecture_fingerprint",
            "backbone_schema_fingerprint",
        )
    ):
        return (
            globals()["create_wavestarnet"],
            globals()["architecture_fingerprint"],
            globals()["backbone_schema_fingerprint"],
        )
    from agrivision4.models.wavestarnet_family import (
        architecture_fingerprint,
        backbone_schema_fingerprint,
        create_wavestarnet,
    )

    return create_wavestarnet, architecture_fingerprint, backbone_schema_fingerprint


def state_schema_sha256(model: nn.Module) -> str:
    schema = [
        (name, list(tensor.shape), str(tensor.dtype))
        for name, tensor in sorted(model.state_dict().items())
    ]
    payload = json.dumps(schema, separators=(",", ":"), sort_keys=False).encode()
    return sha256_bytes(payload)


def build_model(
    config: ScratchConfig,
    device: torch.device,
) -> tuple[nn.Module, dict[str, Any]]:
    spec = model_spec(config.model_key)
    if spec.pretrained:
        raise ValueError("scratch model specification cannot enable pretraining")
    if spec.kind == "wavestarnet":
        factory, architecture_fingerprint, backbone_schema_fingerprint = (
            _wavestar_symbols()
        )
        model = factory(config.model_key, num_classes=config.num_classes)
        architecture_hash = architecture_fingerprint(
            config.model_key, config.num_classes
        )
        backbone_hash = backbone_schema_fingerprint(config.model_key)
        if architecture_hash != EXPECTED_WAVESTAR_ARCHITECTURE_SHA256:
            raise ValueError("WavestarNet architecture SHA-256 mismatch")
        if backbone_hash != EXPECTED_WAVESTAR_BACKBONE_SHA256:
            raise ValueError("WavestarNet backbone schema SHA-256 mismatch")
    else:
        import timm

        if spec.timm_name is None:
            raise ValueError("timm model specification is incomplete")
        model = timm.create_model(
            spec.timm_name,
            pretrained=False,
            num_classes=config.num_classes,
        )
        architecture_hash = state_schema_sha256(model)
        backbone_hash = None
    actual_parameters = sum(parameter.numel() for parameter in model.parameters())
    if actual_parameters != spec.expected_parameters:
        raise ValueError(
            f"parameter count mismatch: {actual_parameters} != {spec.expected_parameters}"
        )
    audit = {
        "model_key": spec.key,
        "display_name": spec.display_name,
        "initialization": "random",
        "pretrained": False,
        "checkpoint_loaded": False,
        "teacher_loaded": False,
        "parameters": actual_parameters,
        "architecture_sha256": architecture_hash,
        "backbone_schema_sha256": backbone_hash,
    }
    return model.to(device), audit


def class_weights(
    train_frame: Any,
    device: torch.device,
    *,
    num_classes: int = NUM_CLASSES,
) -> torch.Tensor:
    labels = train_frame["label_id"]
    if hasattr(labels, "tolist"):
        labels = labels.tolist()
    values = class_weight_values(labels, num_classes)
    return torch.tensor(values, dtype=torch.float32, device=device)


def build_optimizer(
    model: nn.Module,
    config: ScratchConfig,
    device: torch.device,
) -> tuple[torch.optim.Optimizer, bool, list[dict[str, Any]]]:
    groups, audit = build_optimizer_groups(
        model,
        learning_rate=config.learning_rate,
        weight_decay=config.weight_decay,
    )
    fused = device.type == "cuda"
    kwargs = {
        "lr": config.learning_rate,
        "betas": (0.9, 0.999),
        "eps": 1e-8,
        "weight_decay": 0.0,
    }
    try:
        optimizer = torch.optim.AdamW(groups, fused=fused, **kwargs)
    except (TypeError, RuntimeError):
        fused = False
        optimizer = torch.optim.AdamW(groups, **kwargs)
    return optimizer, fused, audit


def fixed_label_metrics(
    *,
    sample_ids: Sequence[str],
    y_true: Sequence[int],
    y_pred: Sequence[int],
    probabilities: Any,
    num_classes: int,
) -> dict[str, Any]:
    if len(sample_ids) != len(y_true) or len(y_true) != len(y_pred):
        raise ValueError("metric input lengths differ")
    if len(set(sample_ids)) != len(sample_ids):
        raise ValueError("duplicate sample IDs")
    probability_array = np.asarray(probabilities, dtype=np.float64)
    if probability_array.shape != (len(y_true), num_classes):
        raise ValueError("probability matrix shape is invalid")
    matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    for target, prediction in zip(y_true, y_pred):
        if int(target) not in range(num_classes) or int(prediction) not in range(num_classes):
            raise ValueError("metric label is outside the fixed class range")
        matrix[int(target), int(prediction)] += 1
    per_class = []
    for label_id in range(num_classes):
        true_positive = int(matrix[label_id, label_id])
        false_positive = int(matrix[:, label_id].sum() - true_positive)
        false_negative = int(matrix[label_id, :].sum() - true_positive)
        support = int(matrix[label_id].sum())
        precision = (
            true_positive / (true_positive + false_positive)
            if true_positive + false_positive
            else 0.0
        )
        recall = (
            true_positive / (true_positive + false_negative)
            if true_positive + false_negative
            else 0.0
        )
        f1 = (
            2.0 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )
        per_class.append(
            {
                "label_id": label_id,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "support": support,
            }
        )
    supports = matrix.sum(axis=1).astype(np.float64)
    total = float(supports.sum())
    accuracy = float(np.trace(matrix) / total)
    macro_precision = float(np.mean([row["precision"] for row in per_class]))
    macro_recall = float(np.mean([row["recall"] for row in per_class]))
    macro_f1 = float(np.mean([row["f1"] for row in per_class]))
    weighted_precision = float(
        sum(row["precision"] * row["support"] for row in per_class) / total
    )
    weighted_recall = float(
        sum(row["recall"] * row["support"] for row in per_class) / total
    )
    weighted_f1 = float(
        sum(row["f1"] * row["support"] for row in per_class) / total
    )
    return {
        "accuracy": accuracy,
        "balanced_accuracy": macro_recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "confusion_matrix": matrix.tolist(),
        "per_class": per_class,
    }


def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> tuple[dict[str, Any], dict[str, Any]]:
    model.eval()
    sample_ids: list[str] = []
    component_ids: list[str] = []
    packaged_paths: list[str] = []
    targets: list[int] = []
    logits_parts = []
    with torch.inference_mode():
        for images, labels, batch_ids, batch_components, batch_paths in loader:
            logits = model(images.float().to(device, non_blocking=True)).float()
            if not torch.isfinite(logits).all():
                raise FloatingPointError("non-finite validation logits")
            sample_ids.extend(batch_ids)
            component_ids.extend(batch_components)
            packaged_paths.extend(batch_paths)
            targets.extend(labels.tolist())
            logits_parts.append(logits.cpu())
    logits = torch.cat(logits_parts, dim=0)
    target_tensor = torch.tensor(targets, dtype=torch.long)
    probabilities = torch.softmax(logits, dim=1)
    predictions = probabilities.argmax(dim=1)
    metrics = fixed_label_metrics(
        sample_ids=sample_ids,
        y_true=targets,
        y_pred=predictions.tolist(),
        probabilities=probabilities.numpy(),
        num_classes=NUM_CLASSES,
    )
    metrics["nll"] = float(functional.cross_entropy(logits, target_tensor).item())
    one_hot = functional.one_hot(target_tensor, num_classes=NUM_CLASSES).float()
    metrics["brier"] = float(
        (probabilities - one_hot).square().sum(dim=1).mean().item()
    )
    confidences = probabilities.max(dim=1).values
    correct = predictions.eq(target_tensor)
    ece = 0.0
    boundaries = torch.linspace(0.0, 1.0, 16)
    for lower, upper in zip(boundaries[:-1], boundaries[1:]):
        selected = confidences.gt(lower) & confidences.le(upper)
        if selected.any():
            ece += float(selected.float().mean()) * abs(
                float(correct[selected].float().mean())
                - float(confidences[selected].mean())
            )
    metrics["ece_15bin"] = ece
    payload = {
        "sample_ids": sample_ids,
        "component_ids": component_ids,
        "packaged_paths": packaged_paths,
        "targets": np.asarray(targets, dtype=np.int64),
        "predictions": predictions.numpy(),
        "logits": logits.numpy(),
        "probabilities": probabilities.numpy(),
    }
    return metrics, payload


def should_step_optimizer(
    batch_index: int,
    *,
    total_batches: int,
    accumulation_steps: int,
) -> bool:
    if accumulation_steps < 1:
        raise ValueError("accumulation_steps must be positive")
    if batch_index not in range(1, total_batches + 1):
        raise ValueError("batch_index is outside the epoch")
    return batch_index % accumulation_steps == 0 or batch_index == total_batches


def _accumulation_window_size(
    batch_index: int,
    total_batches: int,
    accumulation_steps: int,
) -> int:
    window_start = ((batch_index - 1) // accumulation_steps) * accumulation_steps + 1
    return min(accumulation_steps, total_batches - window_start + 1)


def finish_amp_optimizer_step(
    *,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    gradient_clip_norm: float,
) -> tuple[float, bool]:
    scaler.unscale_(optimizer)
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip_norm)
    norm_value = float(norm.detach().cpu() if isinstance(norm, torch.Tensor) else norm)
    skipped = not math.isfinite(norm_value)
    if not skipped:
        scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    return norm_value, skipped


def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    device: torch.device,
    config: ScratchConfig,
) -> dict[str, float | int]:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss_sum = 0.0
    sample_count = 0
    correct = 0
    maximum_gradient_norm = 0.0
    skipped_steps = 0
    total_batches = len(loader)
    for batch_index, (images, labels, _, _, _) in enumerate(loader, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        window_size = _accumulation_window_size(
            batch_index,
            total_batches,
            config.gradient_accumulation_steps,
        )
        with torch.amp.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=device.type == "cuda",
        ):
            logits = model(images)
            full_loss = criterion(logits, labels)
            loss = full_loss / window_size
        if not torch.isfinite(full_loss) or not torch.isfinite(logits).all():
            raise FloatingPointError("non-finite training output")
        scaler.scale(loss).backward()
        if should_step_optimizer(
            batch_index,
            total_batches=total_batches,
            accumulation_steps=config.gradient_accumulation_steps,
        ):
            gradient_norm, skipped = finish_amp_optimizer_step(
                model=model,
                optimizer=optimizer,
                scaler=scaler,
                gradient_clip_norm=config.gradient_clip_norm,
            )
            if math.isfinite(gradient_norm):
                maximum_gradient_norm = max(maximum_gradient_norm, gradient_norm)
            skipped_steps += int(skipped)
        batch_size = labels.shape[0]
        loss_sum += float(full_loss.detach().item()) * batch_size
        sample_count += batch_size
        correct += int(logits.detach().argmax(dim=1).eq(labels).sum().item())
    return {
        "loss": loss_sum / sample_count,
        "accuracy": correct / sample_count,
        "gradient_norm": maximum_gradient_norm,
        "amp_nonfinite_steps_skipped": skipped_steps,
        "optimizer_steps": math.ceil(total_batches / config.gradient_accumulation_steps),
    }


def save_validation_evaluation(
    metrics: dict[str, Any],
    payload: dict[str, Any],
    class_names: tuple[str, ...],
    seed_root: Path,
) -> None:
    scalar_metrics = {
        key: value
        for key, value in metrics.items()
        if key not in {"confusion_matrix", "per_class"}
    }
    write_json(scalar_metrics, seed_root / "validation_metrics.json")
    pd.DataFrame(metrics["confusion_matrix"]).to_csv(
        seed_root / "validation_confusion_matrix.csv",
        index=False,
        header=False,
    )
    report = [
        {**row, "class_name": class_names[int(row["label_id"])]}
        for row in metrics["per_class"]
    ]
    write_json(
        {"labels": list(range(NUM_CLASSES)), "classes": report},
        seed_root / "validation_classification_report.json",
    )
    prediction_frame = pd.DataFrame(
        {
            "sample_id": payload["sample_ids"],
            "component_id": payload["component_ids"],
            "packaged_path": payload["packaged_paths"],
            "true_label_id": payload["targets"],
            "predicted_label_id": payload["predictions"],
        }
    )
    for label_id in range(NUM_CLASSES):
        prediction_frame[f"logit_{label_id:02d}"] = payload["logits"][:, label_id]
        prediction_frame[f"prob_{label_id:02d}"] = payload["probabilities"][:, label_id]
    prediction_frame.to_csv(seed_root / "validation_predictions.csv", index=False)


def _make_scaler(device: torch.device) -> torch.amp.GradScaler:
    return torch.amp.GradScaler(
        "cuda",
        enabled=device.type == "cuda",
        init_scale=128.0,
        growth_interval=2000,
    )


def _strict_reload_and_parity(
    *,
    model: nn.Module,
    checkpoint_path: Path,
    fixed_images: torch.Tensor,
    config: ScratchConfig,
    device: torch.device,
) -> bool:
    model.eval()
    with torch.inference_mode():
        expected = model(fixed_images.float()).float().cpu()
    reloaded, _ = build_model(config, device)
    state = torch.load(checkpoint_path, map_location=device, weights_only=True)
    reloaded.load_state_dict(state, strict=True)
    reloaded.eval()
    with torch.inference_mode():
        actual = reloaded(fixed_images.float()).float().cpu()
    parity = torch.allclose(expected, actual, rtol=1e-5, atol=1e-6)
    del reloaded
    if not parity:
        raise ValueError("checkpoint reload logits differ")
    return True


def run_seed(
    *,
    seed: int,
    frames: dict[str, Any],
    class_names: tuple[str, ...],
    device: torch.device,
    weights: torch.Tensor,
    config: ScratchConfig,
    output_root: Path,
) -> dict[str, Any]:
    seed_everything(seed)
    seed_root = output_root / f"seed_{seed}"
    if seed_root.exists():
        shutil.rmtree(seed_root)
    seed_root.mkdir(parents=True)
    total_epochs = epochs_for_mode(config)
    write_json(
        {
            **asdict(config),
            "seed": seed,
            "active_epochs": total_epochs,
            "selection": "validation_macro_f1_then_nll_then_earlier_epoch",
        },
        seed_root / "config.json",
    )
    loaders = build_loaders(frames, seed, config)
    model, initialization_audit = build_model(config, device)
    write_json(initialization_audit, seed_root / "initialization_audit.json")
    optimizer, fused, optimizer_audit = build_optimizer(model, config, device)
    write_json(optimizer_audit, seed_root / "optimizer_groups.json")
    scheduler = WarmupCosineScheduler(
        optimizer,
        total_epochs=total_epochs,
        warmup_epochs=warmup_epochs_for_mode(config),
        min_lr_ratio=config.min_lr_ratio,
    )
    criterion = nn.CrossEntropyLoss(
        weight=weights,
        label_smoothing=config.label_smoothing,
    )
    scaler = _make_scaler(device)
    early_stopping = EarlyStoppingState(
        patience=config.early_stopping_patience,
        min_delta=config.early_stopping_min_delta,
        start_epoch=config.early_stopping_start_epoch,
    )
    checkpoint_path = seed_root / "best_checkpoint.pt"
    best: dict[str, float | int] | None = None
    history = []
    started = time.perf_counter()
    stop_reason = "maximum_epochs"
    for epoch in range(1, total_epochs + 1):
        learning_rates = scheduler.step(epoch)
        train_metrics = train_epoch(
            model,
            loaders["train"],
            criterion,
            optimizer,
            scaler,
            device,
            config,
        )
        validation_metrics, _ = evaluate_model(model, loaders["validation"], device)
        candidate = {
            "macro_f1": validation_metrics["macro_f1"],
            "nll": validation_metrics["nll"],
            "epoch": epoch,
        }
        improved = checkpoint_is_better(candidate, best)
        if improved:
            best = candidate
            torch.save(model.state_dict(), checkpoint_path)
        should_stop = False
        if config.run_mode != "smoke":
            should_stop = early_stopping.update(epoch, validation_metrics["macro_f1"])
        elapsed = time.perf_counter() - started
        average_epoch = elapsed / epoch
        remaining_epochs = max(0, total_epochs - epoch)
        record = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "gradient_norm": train_metrics["gradient_norm"],
            "amp_nonfinite_steps_skipped": train_metrics[
                "amp_nonfinite_steps_skipped"
            ],
            "optimizer_steps": train_metrics["optimizer_steps"],
            "validation_accuracy": validation_metrics["accuracy"],
            "validation_macro_f1": validation_metrics["macro_f1"],
            "validation_nll": validation_metrics["nll"],
            "learning_rate": learning_rates[0],
            "improved_checkpoint": improved,
            "early_stopping_bad_epochs": early_stopping.bad_epochs,
            "elapsed_seconds": elapsed,
            "estimated_remaining_seconds": average_epoch * remaining_epochs,
            "peak_gpu_memory_bytes": (
                torch.cuda.max_memory_allocated() if device.type == "cuda" else 0
            ),
        }
        history.append(record)
        print(
            json.dumps({"event": "epoch_complete", "seed": seed, **record}, sort_keys=True),
            flush=True,
        )
        if should_stop:
            stop_reason = "early_stopping"
            break
    pd.DataFrame(history).to_csv(seed_root / "history.csv", index=False)
    write_json(
        {**early_stopping.as_dict(), "reason": stop_reason, "epochs_run": len(history)},
        seed_root / "early_stopping.json",
    )
    if best is None or not checkpoint_path.is_file():
        raise RuntimeError("training did not produce a validation-selected checkpoint")
    state = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(state, strict=True)
    fixed_images = next(iter(loaders["validation"]))[0][:2].to(device)
    parity = _strict_reload_and_parity(
        model=model,
        checkpoint_path=checkpoint_path,
        fixed_images=fixed_images,
        config=config,
        device=device,
    )
    validation_metrics, validation_payload = evaluate_model(
        model, loaders["validation"], device
    )
    save_validation_evaluation(
        validation_metrics,
        validation_payload,
        class_names,
        seed_root,
    )
    checkpoint_hash = sha256_file(checkpoint_path)
    (seed_root / "best_checkpoint.sha256").write_text(
        checkpoint_hash + "\n", encoding="utf-8"
    )
    result = {
        "seed": seed,
        **{name: float(validation_metrics[name]) for name in METRIC_NAMES},
        "best_epoch": int(best["epoch"]),
        "epochs_run": len(history),
        "stop_reason": stop_reason,
        "checkpoint_sha256": checkpoint_hash,
        "optimizer_fused": fused,
        "checkpoint_reload": True,
        "strict_state_dict": True,
        "logit_parity": parity,
        "wall_seconds": time.perf_counter() - started,
        "peak_gpu_memory_bytes": (
            torch.cuda.max_memory_allocated() if device.type == "cuda" else 0
        ),
    }
    del loaders, optimizer, scaler, model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return result


def aggregate_seed_metrics(results: Sequence[dict[str, Any]]) -> dict[str, Any]:
    summary: dict[str, Any] = {"seed_count": len(results)}
    for metric in METRIC_NAMES:
        values = [float(row[metric]) for row in results]
        mean = statistics.mean(values)
        standard_deviation = statistics.stdev(values) if len(values) > 1 else 0.0
        margin = (
            4.3026527299 * standard_deviation / math.sqrt(len(values))
            if len(values) == 3
            else None
        )
        summary[metric] = {
            "mean": mean,
            "standard_deviation": standard_deviation,
            "confidence_interval_95": (
                [mean - margin, mean + margin] if margin is not None else None
            ),
            "values": values,
        }
    return summary


def environment_payload(device: torch.device) -> dict[str, Any]:
    import timm
    import torchvision

    return {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "timm": timm.__version__,
        "device": str(device),
        "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
        "cuda_runtime": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
    }


def artifact_inventory(root: Path) -> list[dict[str, Any]]:
    inventory_path = root / "artifact_inventory.json"
    return [
        {
            "path": path.relative_to(root).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in sorted(root.rglob("*"))
        if path.is_file() and path != inventory_path
    ]


def validate_artifact_tree(root: Path, active_seed_values: Sequence[int]) -> None:
    required_root = {
        "access_policy.json",
        "class_weights.json",
        "dataset_summary.json",
        "environment.json",
        "model_summary.json",
        "run_manifest.json",
        "validation_seed_metrics.csv",
        "validation_summary.json",
    }
    missing_root = [name for name in required_root if not (root / name).is_file()]
    if missing_root:
        raise ValueError(f"required root artifacts are missing: {missing_root}")
    required_seed = {
        "best_checkpoint.pt",
        "best_checkpoint.sha256",
        "config.json",
        "early_stopping.json",
        "history.csv",
        "initialization_audit.json",
        "optimizer_groups.json",
        "validation_classification_report.json",
        "validation_confusion_matrix.csv",
        "validation_metrics.json",
        "validation_predictions.csv",
    }
    for seed in active_seed_values:
        seed_root = root / f"seed_{seed}"
        missing = [name for name in required_seed if not (seed_root / name).is_file()]
        if missing:
            raise ValueError(f"required seed artifacts are missing: {missing}")
    for path in root.rglob("*"):
        if not path.is_file():
            continue
        relative_parts = [part.casefold() for part in path.relative_to(root).parts]
        if any(part == "test" or part.startswith("test_") for part in relative_parts):
            raise ValueError(f"test artifact is forbidden: {path}")


def _validate_embedded_sources(values: dict[str, Any]) -> None:
    expected_pairs = (
        ("EMBEDDED_MODEL_SOURCE_BYTES", "EXPECTED_MODEL_SOURCE_SHA256"),
        ("EMBEDDED_CORE_SOURCE_BYTES", "EXPECTED_CORE_SOURCE_SHA256"),
        ("EMBEDDED_RUNTIME_SOURCE_BYTES", "EXPECTED_RUNTIME_SOURCE_SHA256"),
    )
    for bytes_key, hash_key in expected_pairs:
        if bytes_key not in values or hash_key not in values:
            raise ValueError(f"embedded source attestation is missing: {bytes_key}")
        if sha256_bytes(values[bytes_key]) != values[hash_key]:
            raise ValueError(f"embedded source SHA-256 mismatch: {bytes_key}")


def main_from_globals(values: dict[str, Any]) -> None:
    _validate_embedded_sources(values)
    config = ScratchConfig(
        model_key=str(values["MODEL_KEY"]),
        run_mode=str(values["RUN_MODE"]),
        seeds=tuple(int(seed) for seed in values["SEEDS"]),
        max_epochs=int(values["MAX_EPOCHS"]),
        warmup_epochs=int(values["WARMUP_EPOCHS"]),
        learning_rate=float(values["LEARNING_RATE"]),
        weight_decay=float(values["WEIGHT_DECAY"]),
        label_smoothing=float(values["LABEL_SMOOTHING"]),
        gradient_clip_norm=float(values["GRADIENT_CLIP_NORM"]),
        gradient_accumulation_steps=int(values["GRADIENT_ACCUMULATION_STEPS"]),
        early_stopping_patience=int(values["EARLY_STOPPING_PATIENCE"]),
        early_stopping_min_delta=float(values["EARLY_STOPPING_MIN_DELTA"]),
        early_stopping_start_epoch=int(values["EARLY_STOPPING_START_EPOCH"]),
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("controlled Kaggle run requires a CUDA GPU")
    output_root = Path("/kaggle/working") / str(values["OUTPUT_SAFE_NAME"])
    if output_root.exists():
        shutil.rmtree(output_root)
    output_root.mkdir(parents=True)
    started_at = utc_now()
    print(
        json.dumps(
            {
                "event": "run_start",
                "model_key": config.model_key,
                "run_mode": config.run_mode,
                "initialization": "random",
            },
            sort_keys=True,
        ),
        flush=True,
    )
    bundle = load_development_bundle(
        "/kaggle/input",
        expected_hashes=dict(values["EXPECTED_MANIFEST_SHA256"]),
    )
    full_frames = bundle.frames
    frames = full_frames
    if config.run_mode == "smoke":
        frames = {
            "train": stratified_subset(full_frames["train"], 2, 42),
            "validation": stratified_subset(full_frames["validation"], 1, 43),
        }
    active_seed_values = active_seeds(config.run_mode, config.seeds)
    write_json(
        {
            "test_manifest_loaded": False,
            "test_loader_created": False,
            "test_metrics_computed": False,
            "selection_source": "fixed_validation_only",
        },
        output_root / "access_policy.json",
    )
    write_json(environment_payload(device), output_root / "environment.json")
    probe_model, model_audit = build_model(config, device)
    write_json(model_audit, output_root / "model_summary.json")
    del probe_model
    gc.collect()
    torch.cuda.empty_cache()
    write_json(
        {
            "protocol_id": PROTOCOL_ID,
            "kernel_id": values["KERNEL_ID"],
            "expected_version": values["EXPECTED_VERSION"],
            "model_key": config.model_key,
            "display_name": model_spec(config.model_key).display_name,
            "run_mode": config.run_mode,
            "initialization": "random",
            "pretrained": False,
            "active_seeds": active_seed_values,
            "dataset_id": DATASET_ID,
            "started_at": started_at,
            "test_accessed": False,
            "source_sha256": {
                "model": values["EXPECTED_MODEL_SOURCE_SHA256"],
                "core": values["EXPECTED_CORE_SOURCE_SHA256"],
                "runtime": values["EXPECTED_RUNTIME_SOURCE_SHA256"],
            },
            "expected_manifest_sha256": values["EXPECTED_MANIFEST_SHA256"],
        },
        output_root / "run_manifest.json",
    )
    write_json(
        {
            "dataset_id": DATASET_ID,
            "class_names": list(bundle.class_names),
            "development_split_counts": {
                split: len(frame) for split, frame in full_frames.items()
            },
            "active_split_counts": {
                split: len(frame) for split, frame in frames.items()
            },
            "manifest_sha256": {
                "train": sha256_file(bundle.root / "manifests/train.csv"),
                "validation": sha256_file(bundle.root / "manifests/validation.csv"),
                "class_mapping": sha256_file(
                    bundle.root / "manifests/class_mapping.json"
                ),
            },
        },
        output_root / "dataset_summary.json",
    )
    weights = class_weights(full_frames["train"], device)
    write_json(
        {
            "method": "sqrt_inverse_frequency_mean_normalized",
            "weighted_sampler": False,
            "values": weights.detach().cpu().tolist(),
        },
        output_root / "class_weights.json",
    )
    results = [
        run_seed(
            seed=seed,
            frames=frames,
            class_names=bundle.class_names,
            device=device,
            weights=weights,
            config=config,
            output_root=output_root,
        )
        for seed in active_seed_values
    ]
    pd.DataFrame(results).to_csv(
        output_root / "validation_seed_metrics.csv", index=False
    )
    summary = aggregate_seed_metrics(results)
    summary.update(
        {
            "seeds": active_seed_values,
            "selection": "validation_macro_f1_then_nll_then_earlier_epoch",
            "test_accessed": False,
        }
    )
    write_json(summary, output_root / "validation_summary.json")
    validate_artifact_tree(output_root, active_seed_values)
    manifest_path = output_root / "run_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest.update({"completed_at": utc_now(), "status": "complete"})
    write_json(manifest, manifest_path)
    write_json(artifact_inventory(output_root), output_root / "artifact_inventory.json")
    print(
        json.dumps({"event": "run_complete", "output": str(output_root)}, sort_keys=True),
        flush=True,
    )

main_from_globals(globals())


{"event": "run_start", "initialization": "random", "model_key": "wavestarnet_big_transfer_v2", "run_mode": "development"}
{"amp_nonfinite_steps_skipped": 0, "early_stopping_bad_epochs": 0, "elapsed_seconds": 134.339216403, "epoch": 1, "estimated_remaining_seconds": 13299.582423897, "event": "epoch_complete", "gradient_norm": 11.23254108428955, "improved_checkpoint": true, "learning_rate": 5.9999999999999995e-05, "optimizer_steps": 225, "peak_gpu_memory_bytes": 2613857280, "seed": 42, "train_accuracy": 0.020589872008903727, "train_loss": 4.781387034561081, "validation_accuracy": 0.04035874439461883, "validation_macro_f1": 0.017474195000146336, "validation_nll": 4.506685256958008}
{"amp_nonfinite_steps_skipped": 0, "early_stopping_bad_epochs": 0, "elapsed_seconds": 242.314355768, "epoch": 2, "estimated_remaining_seconds": 11873.403432632, "event": "epoch_complete", "gradient_norm": 10.103521347045898, "improved_checkpoint": true, "learning_rate": 0.00011999999999999999, "optimizer_steps"

/tmp/ipykernel_23/3425626651.py:730: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  prediction_frame[f"logit_{label_id:02d}"] = payload["logits"][:, label_id]
/tmp/ipykernel_23/3425626651.py:731: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  prediction_frame[f"prob_{label_id:02d}"] = payload["probabilities"][:, label_id]
/tmp/ipykernel_23/3425626651.py:730: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns

{"amp_nonfinite_steps_skipped": 0, "early_stopping_bad_epochs": 0, "elapsed_seconds": 108.02216501300063, "epoch": 1, "estimated_remaining_seconds": 10694.194336287062, "event": "epoch_complete", "gradient_norm": 10.3718900680542, "improved_checkpoint": true, "learning_rate": 5.9999999999999995e-05, "optimizer_steps": 225, "peak_gpu_memory_bytes": 2614184960, "seed": 2024, "train_accuracy": 0.02003338898163606, "train_loss": 4.783057093155935, "validation_accuracy": 0.04035874439461883, "validation_macro_f1": 0.01968701821129045, "validation_nll": 4.510036945343018}
{"amp_nonfinite_steps_skipped": 0, "early_stopping_bad_epochs": 0, "elapsed_seconds": 216.5659831840003, "epoch": 2, "estimated_remaining_seconds": 10611.733176016014, "event": "epoch_complete", "gradient_norm": 9.594048500061035, "improved_checkpoint": true, "learning_rate": 0.00011999999999999999, "optimizer_steps": 225, "peak_gpu_memory_bytes": 2614184960, "seed": 2024, "train_accuracy": 0.039371174179187535, "train_loss

/tmp/ipykernel_23/3425626651.py:730: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  prediction_frame[f"logit_{label_id:02d}"] = payload["logits"][:, label_id]
/tmp/ipykernel_23/3425626651.py:731: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  prediction_frame[f"prob_{label_id:02d}"] = payload["probabilities"][:, label_id]
/tmp/ipykernel_23/3425626651.py:730: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns

{"amp_nonfinite_steps_skipped": 0, "early_stopping_bad_epochs": 0, "elapsed_seconds": 109.87579436799933, "epoch": 1, "estimated_remaining_seconds": 10877.703642431934, "event": "epoch_complete", "gradient_norm": 11.148164749145508, "improved_checkpoint": true, "learning_rate": 5.9999999999999995e-05, "optimizer_steps": 225, "peak_gpu_memory_bytes": 2614840320, "seed": 3407, "train_accuracy": 0.02003338898163606, "train_loss": 4.776462622330994, "validation_accuracy": 0.04356181934657271, "validation_macro_f1": 0.021857734194261076, "validation_nll": 4.50980806350708}
{"amp_nonfinite_steps_skipped": 0, "early_stopping_bad_epochs": 0, "elapsed_seconds": 217.98356408600011, "epoch": 2, "estimated_remaining_seconds": 10681.194640214006, "event": "epoch_complete", "gradient_norm": 10.636638641357422, "improved_checkpoint": true, "learning_rate": 0.00011999999999999999, "optimizer_steps": 225, "peak_gpu_memory_bytes": 2614840320, "seed": 3407, "train_accuracy": 0.04493600445186422, "train_l

/tmp/ipykernel_23/3425626651.py:730: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  prediction_frame[f"logit_{label_id:02d}"] = payload["logits"][:, label_id]
/tmp/ipykernel_23/3425626651.py:731: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  prediction_frame[f"prob_{label_id:02d}"] = payload["probabilities"][:, label_id]
/tmp/ipykernel_23/3425626651.py:730: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns

{"event": "run_complete", "output": "/kaggle/working/plantwild_scratch_wavestarnet_transfer_v2_development_v01"}
